# 1. Multimodal input preparation

In this notebook, build the multimodal modelling pipeline in a controlled and auditable way.

use the **peer-reviewed interaction pathway architecture as the main backbone**. later evaluate the sparse-attention and masking ideas from sparse-attention variant as optional experimental variants rather than assuming that they are automatically better.

The intended modelling structure is:

1. modality-specific encoders;
2. a interaction pathway cascaded cross-attention path;
3. independent evidential heads for each modality;
4. evidence pathway/Dempster-Shafer fusion of modality opinions;
5. a joint evidential head after multimodal interaction;
6. controlled sparse-attention variant-inspired ablations for sparse attention and additional masking.

This notebook begins with a clean and reproducible data-loading stage. not rebuild baseline alignment inside the modelling notebook. Instead, treat the previously created **baseline-aligned modality manifests** as the source of truth.

The authoritative clinical cohort contains 2,199 participants. The MRI storage folders still contain 1,063 fully preprocessed MRI files, but only the 1,057 MRI observations listed in the authoritative baseline-aligned MRI manifest are eligible for modelling.

# 2. Mount Google Drive

I mount Google Drive before defining any project paths.

If Colab reports that the mountpoint already contains files, restart the runtime and mount Drive again before running the remaining cells. not delete a live Drive mount.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 3. Import libraries and define reproducibility settings

I import only the libraries needed for setup, validation, data loading, and later PyTorch development.

I set a fixed random seed so that data splits, masking experiments, and model initialisation can be reproduced. I also record the active GPU because this notebook is expected to use an NVIDIA A100 when available.

In [ ]:
# Standard library
import os
import re
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Data handling
import numpy as np
import pandas as pd

# Deep learning
import torch

warnings.filterwarnings("default")

# I use one global seed throughout the notebook.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# I prefer deterministic behaviour while establishing the pipeline.
# I may relax this later only if a specific operation requires it.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 4. Define project, cohort, manifest, source, and output paths

I separate four different kinds of paths:

1. the authoritative clinical cohort;
2. the baseline-aligned modality manifests used for modelling;
3. the original cleaned longitudinal sources retained for provenance and auditing;
4. the processed MRI storage folders.

The key modelling rule is:

> I select participants and observations from the baseline-aligned manifests, not by scanning every file or every longitudinal row.

The 1,063 MRI files remain in storage, but the 1,057-row MRI manifest determines which MRI observations may enter the model.

In [ ]:
# ---------------------------------------------------------------------
# Main project roots
# ---------------------------------------------------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "adni_mri"
NON_IMAGING_ROOT = PROJECT_ROOT / "adni_non_imaging"

# ---------------------------------------------------------------------
# Authoritative clinical cohort
# ---------------------------------------------------------------------
AUTHORITATIVE_COHORT_DIR = (
    NON_IMAGING_ROOT
    / "manifests"
    / "authoritative_clinical_cohort"
)

COHORT_PATH = (
    AUTHORITATIVE_COHORT_DIR
    / "authoritative_four_group_clinical_cohort.csv"
)

# ---------------------------------------------------------------------
# Baseline-aligned modality manifests
# ---------------------------------------------------------------------
BASELINE_ALIGNED_ROOT = (
    NON_IMAGING_ROOT
    / "manifests"
    / "baseline_aligned_modalities"
)

MRI_MANIFEST_PATH = (
    BASELINE_ALIGNED_ROOT
    / "mri"
    / "mri_baseline_aligned_authoritative_cohort_1057.csv"
)

PTDEMOG_MANIFEST_PATH = (
    BASELINE_ALIGNED_ROOT
    / "ptdemog"
    / "ptdemog_baseline_aligned_cohort.csv"
)

# ---------------------------------------------------------------------
# Processed MRI storage
# These folders contain the complete preprocessed MRI pool.
# They are not, by themselves, the modelling cohort.
# ---------------------------------------------------------------------
MRI_PROCESSED_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99"
)

MRI_NII_DIR = MRI_PROCESSED_ROOT / "nii_t1"
MRI_NPY_DIR = MRI_PROCESSED_ROOT / "npy_t1"

# ---------------------------------------------------------------------
# Original cleaned longitudinal sources
# I keep these paths for provenance, feature verification, and auditing.
# I do not use these files to repeat baseline selection.
# ---------------------------------------------------------------------
SOURCE_PATHS = {
    "ptdemog_longitudinal": (
        NON_IMAGING_ROOT
        / "interim"
        / "ptdemog_cleaned_longitudinal.csv"
    ),
    "mmse_primary_longitudinal": (
        NON_IMAGING_ROOT
        / "processed"
        / "mmse"
        / "mmse_longitudinal_primary_features_v2.csv"
    ),
    "mmse_optional_longitudinal": (
        NON_IMAGING_ROOT
        / "processed"
        / "mmse"
        / "mmse_longitudinal_optional_domain_features_v2.csv"
    ),
    "faq_longitudinal": (
        NON_IMAGING_ROOT
        / "processed"
        / "faq_model_ready_dated_longitudinal.csv"
    ),
    "adas_longitudinal": (
        NON_IMAGING_ROOT
        / "processed"
        / "adas_clean_longitudinal_interim.csv"
    ),
    "csf_visit_level": (
        NON_IMAGING_ROOT
        / "interim"
        / "csf_core_biomarkers"
        / "csf_core_biomarkers_cleaned_visit_level.csv"
    ),
    "plasma_longitudinal": (
        NON_IMAGING_ROOT
        / "processed"
        / "plasma"
        / "plasma_biomarkers_cleaned_longitudinal.csv"
    ),
    "apoe_participant_level": (
        NON_IMAGING_ROOT
        / "processed"
        / "apoe"
        / "apoe_genotype_cleaned_participant_level.csv"
    ),
}

# ---------------------------------------------------------------------
# Outputs created by this modelling notebook
# ---------------------------------------------------------------------
MODEL_ROOT = PROJECT_ROOT / "models" / "3mt_tmc_evidential"
MANIFEST_OUTPUT_DIR = MODEL_ROOT / "manifests"
CHECKPOINT_DIR = MODEL_ROOT / "checkpoints"
RESULTS_DIR = MODEL_ROOT / "results"
FIGURES_DIR = MODEL_ROOT / "figures"
LOGS_DIR = MODEL_ROOT / "logs"

for directory in [
    MODEL_ROOT,
    MANIFEST_OUTPUT_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Authoritative cohort:", COHORT_PATH)
print("Baseline-aligned manifest root:", BASELINE_ALIGNED_ROOT)
print("MRI modelling manifest:", MRI_MANIFEST_PATH)
print("Model output root:", MODEL_ROOT)

# 5. Validate the required project structure

Before loading any data, The notebook confirms that the authoritative cohort, baseline-aligned manifest root, MRI manifest, PTDEMOG manifest, and MRI storage folders exist.

fail early when a required path is missing. This prevents me from silently training on the wrong folder or on an incomplete copy of the data.

In [ ]:
REQUIRED_PATHS = {
    "authoritative_cohort": COHORT_PATH,
    "baseline_aligned_manifest_root": BASELINE_ALIGNED_ROOT,
    "MRI_baseline_aligned_manifest": MRI_MANIFEST_PATH,
    "PTDEMOG_baseline_aligned_manifest": PTDEMOG_MANIFEST_PATH,
    "MRI_NIfTI_directory": MRI_NII_DIR,
    "MRI_NumPy_directory": MRI_NPY_DIR,
}

path_validation = pd.DataFrame(
    [
        {
            "SOURCE": source,
            "PATH": str(path),
            "EXISTS": path.exists(),
            "TYPE": (
                "directory"
                if path.is_dir()
                else "file"
                if path.is_file()
                else "missing"
            ),
        }
        for source, path in REQUIRED_PATHS.items()
    ]
)

display(path_validation)

missing_paths = path_validation.loc[
    ~path_validation["EXISTS"],
    ["SOURCE", "PATH"],
]

if not missing_paths.empty:
    raise FileNotFoundError(
        "Required cohort or manifest paths were not found:\n\n"
        + missing_paths.to_string(index=False)
    )

print("All required cohort and baseline-aligned manifest paths were found.")

# 6. Discover every baseline-aligned modality manifest

I do not guess the exact filenames for MMSE, FAQ, ADAS, CSF, plasma, or APOE. Instead, I discover every CSV stored under the baseline-aligned modality directory.

This gives me a transparent inventory of the files that were created during preprocessing. review this table before using the manifests in the model.

In [ ]:
manifest_paths = sorted(BASELINE_ALIGNED_ROOT.rglob("*.csv"))

if len(manifest_paths) == 0:
    raise RuntimeError(
        f"No CSV manifests were found under:\n{BASELINE_ALIGNED_ROOT}"
    )

manifest_file_inventory = pd.DataFrame(
    [
        {
            "MODALITY_FOLDER": path.parent.name,
            "FILE_NAME": path.name,
            "PATH": str(path),
        }
        for path in manifest_paths
    ]
)

print(
    f"Found {len(manifest_file_inventory):,} baseline-aligned manifest files."
)

display(manifest_file_inventory)

# 7. Define a conservative ADNI CSV loader

This notebook uses one loader for the authoritative cohort and all baseline-aligned manifests.

The loader performs only conservative operations:

- it strips whitespace from column names;
- it converts `RID` to a nullable integer;
- it standardises common text identifiers;
- it parses columns that clearly represent dates;
- it does not remove participants;
- it does not impute missing values;
- it does not select modelling features;
- it does not alter class labels.

This keeps data loading separate from modelling decisions.

In [ ]:
def load_adni_csv(
    path: Path,
    source_name: str,
    low_memory: bool = False,
) -> pd.DataFrame:
    """
    I load one ADNI CSV and apply only conservative type normalisation.

    I deliberately avoid filtering, feature selection, label changes,
    and missing-value imputation in this function.
    """
    if not path.is_file():
        raise FileNotFoundError(
            f"{source_name} file was not found:\n{path}"
        )

    df = pd.read_csv(path, low_memory=low_memory)
    df.columns = df.columns.astype(str).str.strip()

    if "RID" in df.columns:
        df["RID"] = (
            pd.to_numeric(df["RID"], errors="coerce")
            .astype("Int64")
        )

    for column in ["PTID", "PHASE", "VISCODE", "VISCODE2"]:
        if column in df.columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
            )

    # I parse only columns whose names clearly indicate a date.
    date_name_pattern = re.compile(
        r"(DATE|VISDATE|EXAMDATE|APTESTDT)",
        flags=re.IGNORECASE,
    )

    for column in df.columns:
        if date_name_pattern.search(column):
            parsed = pd.to_datetime(df[column], errors="coerce")

            # I replace the column only when at least one valid date exists.
            if parsed.notna().any():
                df[column] = parsed

    unique_rid = (
        df["RID"].nunique(dropna=True)
        if "RID" in df.columns
        else "RID absent"
    )

    print(
        f"{source_name:<55} "
        f"rows={len(df):>7,} | "
        f"columns={df.shape[1]:>3} | "
        f"unique RID={unique_rid}"
    )

    return df

# 8. Load the authoritative cohort and all baseline-aligned manifests

The notebook loads the authoritative cohort first. I then load every baseline-aligned modality manifest discovered in Section 6.

The notebook preserves each manifest in a dictionary using a stable key based on its folder and filename. This avoids variable-name collisions and lets me inspect every modality consistently.

In [ ]:
cohort_df = load_adni_csv(
    COHORT_PATH,
    source_name="Authoritative clinical cohort",
)

baseline_manifest_dfs: Dict[str, pd.DataFrame] = {}

for path in manifest_paths:
    relative_path = path.relative_to(BASELINE_ALIGNED_ROOT)

    manifest_key = (
        str(relative_path)
        .replace("/", "__")
        .replace("\\", "__")
        .removesuffix(".csv")
    )

    baseline_manifest_dfs[manifest_key] = load_adni_csv(
        path,
        source_name=manifest_key,
    )

print(
    f"\nLoaded {len(baseline_manifest_dfs):,} "
    "baseline-aligned modality manifests."
)

# 9. Inspect the authoritative cohort and confirm the four clinical groups

The notebook confirms the cohort size and inspect likely label columns.

The expected authoritative cohort contains 2,199 participants:

- CN: 1,196;
- AD: 459;
- sMCI: 295;
- pMCI: 249.

I do not hard-code the label-column name before inspecting the actual file.

In [ ]:
print(f"Authoritative cohort rows: {len(cohort_df):,}")
print(f"Authoritative cohort columns: {cohort_df.shape[1]:,}")

if "RID" not in cohort_df.columns:
    raise KeyError(
        "The authoritative cohort does not contain an RID column."
    )

print(
    "Unique authoritative RIDs:",
    f"{cohort_df['RID'].nunique(dropna=True):,}",
)

candidate_label_columns = [
    column
    for column in cohort_df.columns
    if any(
        token in column.upper()
        for token in [
            "GROUP",
            "LABEL",
            "CLASS",
            "DIAG",
            "COHORT",
            "OUTCOME",
        ]
    )
]

print("\nPossible label columns:")
print(candidate_label_columns)

for column in candidate_label_columns:
    if cohort_df[column].nunique(dropna=True) <= 20:
        print(f"\nDistribution for {column}:")
        display(
            cohort_df[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="COUNT")
        )

EXPECTED_AUTHORITATIVE_ROWS = 2199

if len(cohort_df) != EXPECTED_AUTHORITATIVE_ROWS:
    raise ValueError(
        "Unexpected authoritative cohort size. "
        f"Expected {EXPECTED_AUTHORITATIVE_ROWS:,}, "
        f"found {len(cohort_df):,}."
    )

if cohort_df["RID"].nunique(dropna=True) != EXPECTED_AUTHORITATIVE_ROWS:
    raise ValueError(
        "The authoritative cohort should contain one row per participant, "
        "but the row count and unique-RID count differ."
    )

print("\nThe authoritative cohort passed its participant-level size check.")

# 10. Build a manifest-level data inventory

I summarise the number of rows, columns, participants, missing identifiers, duplicate identifiers, and fully duplicated rows in every baseline-aligned manifest.

A duplicate RID is not automatically treated as an error in this summary. inspect the modality and its intended structure. However, participant-level baseline manifests are generally expected to contain one selected row per participant.

In [ ]:
manifest_inventory_rows = []

for manifest_name, df in baseline_manifest_dfs.items():
    manifest_inventory_rows.append(
        {
            "MANIFEST": manifest_name,
            "ROWS": len(df),
            "COLUMNS": df.shape[1],
            "UNIQUE_RID": (
                df["RID"].nunique(dropna=True)
                if "RID" in df.columns
                else pd.NA
            ),
            "MISSING_RID": (
                int(df["RID"].isna().sum())
                if "RID" in df.columns
                else pd.NA
            ),
            "ROWS_INVOLVED_IN_DUPLICATED_RID": (
                int(df["RID"].duplicated(keep=False).sum())
                if "RID" in df.columns
                else pd.NA
            ),
            "DUPLICATED_FULL_ROWS": int(df.duplicated().sum()),
        }
    )

manifest_inventory_df = (
    pd.DataFrame(manifest_inventory_rows)
    .sort_values("MANIFEST")
    .reset_index(drop=True)
)

display(manifest_inventory_df)

# 11. Verify that every modality manifest is a subset of the authoritative cohort

The authoritative cohort is the master participant set.

For every modality manifest, The notebook verifies that all listed RIDs belong to the authoritative cohort. A modality may cover only a subset of the 2,199 participants, but it must never introduce an outside participant.

In [ ]:
authoritative_rids = set(
    cohort_df["RID"].dropna().astype(int)
)

cohort_consistency_rows = []

for manifest_name, df in baseline_manifest_dfs.items():
    if "RID" not in df.columns:
        cohort_consistency_rows.append(
            {
                "MANIFEST": manifest_name,
                "ROWS": len(df),
                "UNIQUE_RID": pd.NA,
                "RID_OUTSIDE_AUTHORITATIVE_COHORT": pd.NA,
                "IS_SUBSET": False,
                "NOTE": "RID column absent",
            }
        )
        continue

    manifest_rids = set(
        df["RID"].dropna().astype(int)
    )

    outside_rids = manifest_rids - authoritative_rids

    cohort_consistency_rows.append(
        {
            "MANIFEST": manifest_name,
            "ROWS": len(df),
            "UNIQUE_RID": len(manifest_rids),
            "RID_OUTSIDE_AUTHORITATIVE_COHORT": len(outside_rids),
            "IS_SUBSET": len(outside_rids) == 0,
            "NOTE": (
                ""
                if len(outside_rids) == 0
                else f"Example outside RIDs: {sorted(outside_rids)[:10]}"
            ),
        }
    )

cohort_consistency_df = (
    pd.DataFrame(cohort_consistency_rows)
    .sort_values("MANIFEST")
    .reset_index(drop=True)
)

display(cohort_consistency_df)

invalid_manifests = cohort_consistency_df.loc[
    ~cohort_consistency_df["IS_SUBSET"]
]

if not invalid_manifests.empty:
    raise ValueError(
        "At least one baseline-aligned modality manifest is not a "
        "valid subset of the authoritative cohort."
    )

print(
    "All baseline-aligned modality manifests are valid subsets "
    "of the authoritative clinical cohort."
)

# 12. Load and validate the 1,057-participant MRI manifest

load the MRI baseline-aligned manifest directly.

This manifest is the modelling source of truth for MRI. I do not use the total number of files in the MRI storage folders to define the MRI cohort.

The expected result is:

- 1,057 rows;
- 1,057 unique RIDs;
- no missing RIDs;
- no duplicated RIDs;
- every RID contained in the authoritative cohort.

In [ ]:
mri_manifest_df = load_adni_csv(
    MRI_MANIFEST_PATH,
    source_name="MRI baseline-aligned authoritative manifest",
)

print("\nMRI manifest columns:")
print(mri_manifest_df.columns.tolist())

if "RID" not in mri_manifest_df.columns:
    raise KeyError(
        "The MRI baseline-aligned manifest does not contain RID."
    )

mri_rows = len(mri_manifest_df)
mri_unique_rids = mri_manifest_df["RID"].nunique(dropna=True)
mri_missing_rids = int(mri_manifest_df["RID"].isna().sum())
mri_duplicated_rid_rows = int(
    mri_manifest_df["RID"].duplicated(keep=False).sum()
)

print(f"\nMRI manifest rows: {mri_rows:,}")
print(f"Unique MRI RIDs: {mri_unique_rids:,}")
print(f"Missing MRI RIDs: {mri_missing_rids:,}")
print(
    "Rows involved in duplicated RIDs:",
    f"{mri_duplicated_rid_rows:,}",
)

EXPECTED_MRI_MANIFEST_ROWS = 1057

if mri_rows != EXPECTED_MRI_MANIFEST_ROWS:
    raise ValueError(
        "Unexpected MRI manifest size. "
        f"Expected {EXPECTED_MRI_MANIFEST_ROWS:,}, "
        f"found {mri_rows:,}."
    )

if mri_unique_rids != EXPECTED_MRI_MANIFEST_ROWS:
    raise ValueError(
        "The MRI manifest should contain one selected row per participant, "
        "but its row count and unique-RID count differ."
    )

if mri_missing_rids != 0:
    raise ValueError(
        "The MRI manifest contains missing RIDs."
    )

if mri_duplicated_rid_rows != 0:
    raise ValueError(
        "The MRI manifest contains duplicated participant RIDs."
    )

mri_rids = set(
    mri_manifest_df["RID"].dropna().astype(int)
)

outside_authoritative = mri_rids - authoritative_rids

if outside_authoritative:
    raise ValueError(
        "The MRI manifest contains RIDs outside the authoritative cohort. "
        f"Examples: {sorted(outside_authoritative)[:10]}"
    )

print(
    "\nThe 1,057-row MRI manifest passed all participant-level checks."
)

# 13. Identify MRI file-reference columns in the manifest

The notebook inspects the MRI manifest for columns that may contain `.npy`, `.nii`, `.nii.gz`, filename, image, or path information.

I do not assume the column name. This keeps the notebook compatible with the manifest that was actually saved during preprocessing.

In [ ]:
possible_mri_path_columns = []

for column in mri_manifest_df.columns:
    values = mri_manifest_df[column].dropna().astype(str)

    if values.empty:
        continue

    lower_values = values.str.lower()

    contains_npy = lower_values.str.endswith(".npy").any()
    contains_nifti = (
        lower_values.str.endswith(".nii").any()
        or lower_values.str.endswith(".nii.gz").any()
    )

    name_suggests_reference = any(
        token in column.upper()
        for token in [
            "PATH",
            "FILE",
            "NPY",
            "NII",
            "IMAGE",
        ]
    )

    if contains_npy or contains_nifti or name_suggests_reference:
        possible_mri_path_columns.append(
            {
                "COLUMN": column,
                "CONTAINS_NPY": contains_npy,
                "CONTAINS_NIFTI": contains_nifti,
                "NAME_SUGGESTS_FILE_REFERENCE": name_suggests_reference,
                "NON_MISSING": int(values.shape[0]),
                "EXAMPLE": values.iloc[0],
            }
        )

possible_mri_path_columns_df = pd.DataFrame(
    possible_mri_path_columns
)

display(possible_mri_path_columns_df)

# 14. Resolve exactly one NumPy MRI file for every manifest row

I resolve `.npy` model inputs from the 1,057-row MRI manifest.

The resolver follows a strict order:

1. use an explicit `.npy` path or filename from the manifest when available;
2. otherwise match MRI files using exact PTID or RID tokens;
3. reject unmatched or ambiguous cases instead of choosing silently.

The storage directory may contain 1,063 files, but only files linked to the 1,057 manifest rows are retained.

In [ ]:
# ---------------------------------------------------------------------
# I use the final normalised T1 NumPy volume as the MRI model input.
# The manifest contains several intermediate and derived MRI products,
# but only normalized_t1_npy_path is intended for the T1 model branch.
# ---------------------------------------------------------------------
MRI_MODEL_PATH_COLUMN = "normalized_t1_npy_path"

if MRI_MODEL_PATH_COLUMN not in mri_manifest_df.columns:
    raise KeyError(
        f"The MRI manifest does not contain {MRI_MODEL_PATH_COLUMN!r}."
    )

mri_model_df = mri_manifest_df.copy()

mri_model_df["NPY_PATH"] = (
    mri_model_df[MRI_MODEL_PATH_COLUMN]
    .astype("string")
    .str.strip()
)

# I verify that every manifest row contains a model-input path.
missing_path_values = mri_model_df["NPY_PATH"].isna().sum()

print(f"Missing path values: {missing_path_values:,}")

if missing_path_values != 0:
    raise ValueError(
        "Some MRI manifest rows do not contain a normalised T1 NumPy path."
    )

# I convert each stored path to Path and test whether the file exists.
mri_model_df["NPY_PATH_EXISTS"] = mri_model_df["NPY_PATH"].map(
    lambda value: Path(value).is_file()
)

existence_summary = (
    mri_model_df["NPY_PATH_EXISTS"]
    .value_counts(dropna=False)
    .rename_axis("NPY_PATH_EXISTS")
    .reset_index(name="ROWS")
)

display(existence_summary)

missing_files_df = mri_model_df.loc[
    ~mri_model_df["NPY_PATH_EXISTS"],
    ["RID", "PTID", MRI_MODEL_PATH_COLUMN],
]

if not missing_files_df.empty:
    display(missing_files_df.head(30))

    raise FileNotFoundError(
        f"{len(missing_files_df):,} normalised T1 NumPy paths "
        "from the MRI manifest do not currently resolve to files."
    )

# I also confirm that no model-input file is assigned to two participants.
duplicated_path_rows = mri_model_df.loc[
    mri_model_df["NPY_PATH"].duplicated(keep=False),
    ["RID", "PTID", "NPY_PATH"],
]

if not duplicated_path_rows.empty:
    display(duplicated_path_rows)

    raise ValueError(
        "At least one normalised T1 NumPy path is assigned to more than "
        "one participant."
    )

print(
    f"Successfully validated one final normalised T1 NumPy file for all "
    f"{len(mri_model_df):,} eligible MRI participants."
)

# 15. Validate MRI array structure without loading the full MRI cohort into RAM

The notebook loads only one resolved MRI array using memory mapping.

I confirm:

- the expected shape of `177 × 213 × 183`;
- a numeric data type;
- finite values;
- the absence of NaNs and infinities.

The future PyTorch `Dataset` will load MRI arrays per sample rather than placing all volumes in memory at once.

In [ ]:
example_mri_path = Path(mri_model_df.iloc[0]["NPY_PATH"])
example_mri = np.load(example_mri_path, mmap_mode="r")

EXPECTED_MRI_SHAPE = (177, 213, 183)

print("Example MRI file:", example_mri_path)
print("Shape:", example_mri.shape)
print("Dtype:", example_mri.dtype)
print("Minimum:", float(np.nanmin(example_mri)))
print("Maximum:", float(np.nanmax(example_mri)))
print("Mean:", float(np.nanmean(example_mri)))
print("Contains NaN:", bool(np.isnan(example_mri).any()))
print("Contains infinity:", bool(np.isinf(example_mri).any()))

if example_mri.shape != EXPECTED_MRI_SHAPE:
    raise ValueError(
        "Unexpected MRI shape. "
        f"Expected {EXPECTED_MRI_SHAPE}, found {example_mri.shape}."
    )

if not np.issubdtype(example_mri.dtype, np.number):
    raise TypeError(
        f"The MRI array is not numeric: {example_mri.dtype}"
    )

if np.isnan(example_mri).any():
    raise ValueError(
        "The example MRI array contains NaN values."
    )

if np.isinf(example_mri).any():
    raise ValueError(
        "The example MRI array contains infinite values."
    )

print("\nThe example MRI array passed the initial structural checks.")

# 16. Save a reproducible source and manifest audit

The notebook saves three small audit files:

1. the required-path validation table;
2. the discovered baseline-manifest inventory;
3. the 1,057-row MRI-to-NumPy resolution table.

These files record exactly which inputs were visible when this modelling notebook was prepared.

In [ ]:
# ---------------------------------------------------------------------
# I save compact audit files that document the exact inputs used by
# this modelling notebook.
# ---------------------------------------------------------------------

PATH_AUDIT_PATH = (
    MANIFEST_OUTPUT_DIR
    / "required_path_validation.csv"
)

MANIFEST_INVENTORY_PATH = (
    MANIFEST_OUTPUT_DIR
    / "baseline_aligned_manifest_inventory.csv"
)

MRI_RESOLUTION_PATH = (
    MANIFEST_OUTPUT_DIR
    / "mri_authoritative_1057_normalized_t1_paths.csv"
)

# ---------------------------------------------------------------------
# Save the required-path validation table.
# ---------------------------------------------------------------------
path_validation.to_csv(
    PATH_AUDIT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Save the discovered baseline-aligned manifest inventory.
# ---------------------------------------------------------------------
manifest_file_inventory.to_csv(
    MANIFEST_INVENTORY_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Save the final MRI model-input mapping.
#
# I save only the participant identifiers, the original manifest path,
# and the file-existence result. The selected MRI model input is the
# final normalised T1 NumPy volume stored in normalized_t1_npy_path.
# ---------------------------------------------------------------------
required_mri_audit_columns = [
    "RID",
    "PTID",
    MRI_MODEL_PATH_COLUMN,
    "NPY_PATH_EXISTS",
]

missing_audit_columns = [
    column
    for column in required_mri_audit_columns
    if column not in mri_model_df.columns
]

if missing_audit_columns:
    raise KeyError(
        "The MRI audit table is missing required columns: "
        + ", ".join(missing_audit_columns)
    )

mri_model_df[
    required_mri_audit_columns
].to_csv(
    MRI_RESOLUTION_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Confirm that all audit files were written successfully.
# ---------------------------------------------------------------------
saved_audit_paths = {
    "Required path validation": PATH_AUDIT_PATH,
    "Baseline manifest inventory": MANIFEST_INVENTORY_PATH,
    "MRI normalised T1 path mapping": MRI_RESOLUTION_PATH,
}

for audit_name, audit_path in saved_audit_paths.items():
    if not audit_path.is_file():
        raise FileNotFoundError(
            f"{audit_name} was not saved successfully:\n{audit_path}"
        )

    print(f"Saved {audit_name}:")
    print(audit_path)
    print()

print("All source and manifest audit files were saved successfully.")

# 17. Data-loading checkpoint

At this point, I have:

- mounted Google Drive;
- confirmed the A100-compatible PyTorch environment;
- loaded the 2,199-participant authoritative clinical cohort;
- discovered and loaded every baseline-aligned modality manifest;
- confirmed that every modality manifest is a subset of the authoritative cohort;
- loaded and validated the 1,057-participant MRI manifest;
- resolved exactly one `.npy` MRI file for every eligible MRI participant;
- verified one MRI array without loading the full imaging cohort into memory;
- saved reproducible input audits.

I have **not** yet:

- chosen final feature columns;
- encoded categorical variables;
- fitted numerical scalers;
- created train, validation, or test splits;
- applied modality dropout;
- defined the interaction pathway cascade;
- added evidential heads;
- implemented modality-specific evidence fusion;
- tested sparse-attention variant sparse attention or masking.

Those steps should follow only after the exact columns and participant coverage of every baseline-aligned manifest have been reviewed.

# 18. Review the baseline-aligned modality manifests before building the modelling table

Before I merge any modality into a single participant-level dataset, I need to understand exactly what each baseline-aligned manifest contains.

The preprocessing stage has already selected the measurement nearest to the authoritative clinical baseline using the predefined timing rules:

- cognitive and functional measures: within ±90 days;
- CSF and plasma biomarkers: within ±180 days;
- APOE genotype: no timing window, because genotype is stable;
- MRI: the accepted baseline-aligned scan listed in the 1,057-participant MRI manifest.

therefore **not repeat baseline matching in this notebook**. Repeating it could select different rows and make the modelling cohort inconsistent with the preprocessing work already completed.

The next checks will answer four questions for every modality.

### 18.1. Which file is the final modelling manifest?

Some modality folders may contain supporting, quality-control, or comparison files in addition to the final baseline-aligned manifest.

### 18.2. Which columns belong to which role?

I need to distinguish between:

- participant identifiers;
- labels;
- visit and examination dates;
- quality-control fields;
- availability indicators;
- candidate model features;
- administrative or provenance fields.

not treat every numeric column as a model input. Timing offsets, source-row indices, file counts, path-existence flags, and validation columns may be useful for auditing but inappropriate as predictors.

### 18.3. Does each participant appear at most once?

A final baseline-aligned modelling manifest should normally contain one selected observation per participant.

Any duplicated RID must be understood before merging.

### 18.4. How much participant coverage does each modality provide?

Missing modalities will remain explicitly missing.

Participants will not be removed merely because CSF, plasma, MRI, APOE, or a cognitive assessment is unavailable.

The intended structure of the final modelling table is:

$$
\text{one authoritative participant row}
+
\text{selected features from each available modality}
+
\text{one availability indicator per modality}
$$

The master table will retain all 2,199 authoritative participants during construction.

For participant $i$ and modality $m$, modality availability will be represented as:

$$
a_i^{(m)} =
\begin{cases}
1, & \text{if modality } m \text{ is available for participant } i, \\
0, & \text{if modality } m \text{ is unavailable for participant } i.
\end{cases}
$$

At this stage, I am only inspecting schemas and coverage.

I am not yet:

- selecting final predictors;
- fitting numerical transformations;
- encoding categorical variables;
- defining the final experimental target;
- creating training, validation, or test splits.

In [ ]:
# ---------------------------------------------------------------------
# I create a structured schema and coverage summary for every
# baseline-aligned modality manifest.
#
# This cell does not select model features yet. Its purpose is only to
# show me:
#   - how many rows and participants each manifest contains;
#   - whether RID is missing or duplicated;
#   - which columns are present;
#   - the data type and missingness of every column;
#   - a small example value for each column.
#
# I will use this information to distinguish genuine predictors from
# identifiers, dates, quality-control fields, path columns, provenance
# variables, and availability indicators.
# ---------------------------------------------------------------------

manifest_schema_rows = []

for manifest_name, df in baseline_manifest_dfs.items():

    # I record participant-level coverage once for every manifest.
    manifest_row_count = len(df)

    manifest_unique_rids = (
        df["RID"].nunique(dropna=True)
        if "RID" in df.columns
        else pd.NA
    )

    manifest_missing_rids = (
        int(df["RID"].isna().sum())
        if "RID" in df.columns
        else pd.NA
    )

    manifest_duplicated_rid_rows = (
        int(df["RID"].duplicated(keep=False).sum())
        if "RID" in df.columns
        else pd.NA
    )

    # I inspect every column separately.
    for column in df.columns:

        non_missing_values = df[column].dropna()

        # I keep only a short example value so the summary remains readable.
        if non_missing_values.empty:
            example_value = pd.NA
        else:
            example_value = non_missing_values.iloc[0]

            # I shorten long file paths or text fields in the display.
            if isinstance(example_value, str) and len(example_value) > 120:
                example_value = example_value[:117] + "..."

        manifest_schema_rows.append(
            {
                "MANIFEST": manifest_name,
                "MANIFEST_ROWS": manifest_row_count,
                "UNIQUE_RID": manifest_unique_rids,
                "MISSING_RID": manifest_missing_rids,
                "ROWS_INVOLVED_IN_DUPLICATED_RID": (
                    manifest_duplicated_rid_rows
                ),
                "COLUMN": column,
                "DTYPE": str(df[column].dtype),
                "NON_MISSING": int(df[column].notna().sum()),
                "MISSING": int(df[column].isna().sum()),
                "MISSING_PERCENT": round(
                    100 * df[column].isna().mean(),
                    2,
                ),
                "UNIQUE_VALUES": int(
                    df[column].nunique(dropna=True)
                ),
                "EXAMPLE": example_value,
            }
        )

manifest_schema_df = pd.DataFrame(manifest_schema_rows)

# I sort by manifest and original column order as closely as possible.
manifest_schema_df = (
    manifest_schema_df
    .sort_values(
        ["MANIFEST", "COLUMN"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print(
    f"Schema summary created for "
    f"{manifest_schema_df['MANIFEST'].nunique():,} manifests "
    f"and {len(manifest_schema_df):,} manifest columns."
)

display(manifest_schema_df)

# 19. Confirm participant coverage using the correct modality-level availability indicators

The previous coverage calculation was correct for most modalities, but CSF was handled by the fallback logic because its manifest key is `csf_core_biomarkers`, not simply `csf`.

The CSF manifest already contains the correct overall availability indicators:

- `CSF_AVAILABLE`: at least one core CSF biomarker is available within the ±180-day baseline window;
- `CSF_RATIO_AVAILABLE`: the CSF Aβ42/Aβ40 ratio is available;
- `ABETA42_40_RATIO_AVAILABLE`: a feature-specific ratio availability flag;
- `AVAILABLE_CORE_BIOMARKER_COUNT`: the number of available core CSF biomarkers.

For overall modality coverage, I must use `CSF_AVAILABLE`.

This gives:

- CSF available: 1,225 participants;
- CSF unavailable: 974 participants;
- CSF coverage: 55.71%;
- CSF ratio available: 366 participants.

The ratio indicator is not a replacement for the overall CSF indicator. A participant may have valid CSF information even when the Aβ42/Aβ40 ratio is unavailable.

therefore use one explicitly defined overall availability column per modality and keep biomarker-specific indicators for later feature-level missingness handling.

In [ ]:
# ---------------------------------------------------------------------
# I calculate participant coverage using an explicitly defined overall
# availability indicator for every baseline-aligned modality.
#
# This version fixes the CSF mapping by using the actual modality key
# "csf_core_biomarkers" and the existing CSF_AVAILABLE column.
#
# I do not infer overall modality availability from biomarker counts
# when an authoritative modality-level indicator already exists.
# ---------------------------------------------------------------------

AUTHORITATIVE_PARTICIPANT_COUNT = len(cohort_df)

# ---------------------------------------------------------------------
# I map each manifest folder name to its authoritative modality-level
# availability column.
#
# MRI is handled separately because its manifest contains only the
# 1,057 participants with an accepted baseline-aligned MRI.
#
# PTDEMOG does not currently require a separate availability flag
# because its baseline-aligned manifest contains all 2,199 participants.
# ---------------------------------------------------------------------
PREFERRED_AVAILABILITY_COLUMNS = {
    "adas": "ADAS_AVAILABLE",
    "apoe": "APOE_AVAILABLE",
    "csf_core_biomarkers": "CSF_AVAILABLE",
    "faq": "FAQ_AVAILABLE",
    "mmse": "MMSE_AVAILABLE",
    "plasma": "PLASMA_AVAILABLE",
}


def normalise_boolean_indicator(
    series: pd.Series,
    column_name: str,
) -> pd.Series:
    """
    I convert a stored availability indicator into a Boolean series.

    I support Boolean, numeric, and common text representations. I raise
    an error if unexpected non-missing values are present because an
    availability column should have a simple binary interpretation.
    """
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        numeric_values = pd.to_numeric(
            series,
            errors="coerce",
        )

        unexpected_numeric_values = set(
            numeric_values.dropna().unique()
        ) - {0, 1, 0.0, 1.0}

        if unexpected_numeric_values:
            raise ValueError(
                f"{column_name} contains unexpected numeric values: "
                f"{sorted(unexpected_numeric_values)}"
            )

        return numeric_values.fillna(0).eq(1)

    normalised_text = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    true_values = {
        "true",
        "1",
        "1.0",
        "yes",
        "y",
        "available",
    }

    false_values = {
        "false",
        "0",
        "0.0",
        "no",
        "n",
        "unavailable",
    }

    observed_values = set(
        normalised_text.dropna().unique()
    )

    unexpected_text_values = (
        observed_values
        - true_values
        - false_values
    )

    if unexpected_text_values:
        raise ValueError(
            f"{column_name} contains unexpected text values: "
            f"{sorted(unexpected_text_values)}"
        )

    return normalised_text.isin(true_values)


coverage_rows = []

for manifest_name, df in sorted(
    baseline_manifest_dfs.items()
):
    modality = manifest_name.split("__")[0].lower()

    if "RID" not in df.columns:
        raise KeyError(
            f"{manifest_name} does not contain an RID column."
        )

    unique_rids = int(
        df["RID"].nunique(dropna=True)
    )

    all_availability_related_columns = [
        column
        for column in df.columns
        if (
            "AVAILABLE" in column.upper()
            or "AVAILABILITY" in column.upper()
        )
    ]

    preferred_column = (
        PREFERRED_AVAILABILITY_COLUMNS.get(modality)
    )

    # -------------------------------------------------------------
    # MRI coverage comes directly from the accepted MRI manifest.
    # -------------------------------------------------------------
    if modality == "mri":
        selected_availability_column = (
            "MRI manifest membership"
        )

        available_participants = unique_rids

        coverage_method = (
            "Unique RIDs in the authoritative 1,057-row MRI manifest"
        )

    # -------------------------------------------------------------
    # PTDEMOG contains one row for every authoritative participant.
    # -------------------------------------------------------------
    elif modality == "ptdemog":
        selected_availability_column = (
            "Full authoritative-cohort coverage"
        )

        available_participants = unique_rids

        coverage_method = (
            "All authoritative participants are represented "
            "in the PTDEMOG baseline-aligned manifest"
        )

    # -------------------------------------------------------------
    # All remaining modalities use their explicit overall indicator.
    # -------------------------------------------------------------
    else:
        if preferred_column is None:
            raise KeyError(
                f"No preferred availability column was defined for "
                f"modality {modality!r}."
            )

        if preferred_column not in df.columns:
            raise KeyError(
                f"{manifest_name} does not contain the expected overall "
                f"availability column {preferred_column!r}."
            )

        availability_mask = normalise_boolean_indicator(
            df[preferred_column],
            column_name=preferred_column,
        )

        selected_availability_column = preferred_column

        available_participants = int(
            availability_mask.sum()
        )

        coverage_method = (
            f"Explicit modality-level indicator: {preferred_column}"
        )

    unavailable_participants = (
        AUTHORITATIVE_PARTICIPANT_COUNT
        - available_participants
    )

    coverage_percent = round(
        100
        * available_participants
        / AUTHORITATIVE_PARTICIPANT_COUNT,
        2,
    )

    coverage_rows.append(
        {
            "MODALITY": modality.upper(),
            "MANIFEST": manifest_name,
            "MANIFEST_ROWS": len(df),
            "UNIQUE_RID": unique_rids,
            "SELECTED_AVAILABILITY_COLUMN": (
                selected_availability_column
            ),
            "ALL_AVAILABILITY_RELATED_COLUMNS": (
                ", ".join(all_availability_related_columns)
                if all_availability_related_columns
                else "None"
            ),
            "AVAILABLE_PARTICIPANTS": (
                available_participants
            ),
            "UNAVAILABLE_PARTICIPANTS": (
                unavailable_participants
            ),
            "COVERAGE_PERCENT": coverage_percent,
            "COVERAGE_METHOD": coverage_method,
        }
    )


modality_coverage_df = (
    pd.DataFrame(coverage_rows)
    .sort_values("MODALITY")
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------
# I display the complete table without truncating long text columns.
# ---------------------------------------------------------------------
with pd.option_context(
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    300,
):
    display(modality_coverage_df)

print("\nCOMPACT MODALITY COVERAGE SUMMARY\n")

print(
    modality_coverage_df[
        [
            "MODALITY",
            "AVAILABLE_PARTICIPANTS",
            "UNAVAILABLE_PARTICIPANTS",
            "COVERAGE_PERCENT",
            "SELECTED_AVAILABILITY_COLUMN",
        ]
    ].to_string(index=False)
)

# ---------------------------------------------------------------------
# I validate the known coverage counts already established during
# preprocessing. A mismatch now would indicate either a wrong manifest,
# a wrong availability column, or an unexpected file change.
# ---------------------------------------------------------------------
EXPECTED_AVAILABLE_COUNTS = {
    "ADAS": 2096,
    "APOE": 2126,
    "CSF_CORE_BIOMARKERS": 1225,
    "FAQ": 2098,
    "MMSE": 1828,
    "MRI": 1057,
    "PLASMA": 841,
    "PTDEMOG": 2199,
}

coverage_check_rows = []

for modality, expected_count in EXPECTED_AVAILABLE_COUNTS.items():
    observed_rows = modality_coverage_df.loc[
        modality_coverage_df["MODALITY"] == modality
    ]

    if len(observed_rows) != 1:
        raise ValueError(
            f"Expected exactly one coverage row for {modality}, "
            f"but found {len(observed_rows)}."
        )

    observed_count = int(
        observed_rows.iloc[0]["AVAILABLE_PARTICIPANTS"]
    )

    coverage_check_rows.append(
        {
            "MODALITY": modality,
            "EXPECTED_AVAILABLE": expected_count,
            "OBSERVED_AVAILABLE": observed_count,
            "MATCHES_EXPECTATION": (
                observed_count == expected_count
            ),
        }
    )

coverage_validation_df = pd.DataFrame(
    coverage_check_rows
)

display(coverage_validation_df)

failed_coverage_checks = coverage_validation_df.loc[
    ~coverage_validation_df["MATCHES_EXPECTATION"]
]

if not failed_coverage_checks.empty:
    raise ValueError(
        "At least one modality coverage count does not match the "
        "validated preprocessing result."
    )

# ---------------------------------------------------------------------
# I separately report CSF ratio coverage because it is an important
# feature-level availability measure, but not the overall CSF indicator.
# ---------------------------------------------------------------------
csf_manifest_name = (
    "csf_core_biomarkers__"
    "csf_core_biomarkers_baseline_aligned_cohort"
)

csf_df = baseline_manifest_dfs[csf_manifest_name]

required_csf_indicator_columns = [
    "CSF_AVAILABLE",
    "CSF_RATIO_AVAILABLE",
]

missing_csf_indicator_columns = [
    column
    for column in required_csf_indicator_columns
    if column not in csf_df.columns
]

if missing_csf_indicator_columns:
    raise KeyError(
        "The CSF manifest is missing expected availability columns: "
        + ", ".join(missing_csf_indicator_columns)
    )

csf_available_count = int(
    normalise_boolean_indicator(
        csf_df["CSF_AVAILABLE"],
        column_name="CSF_AVAILABLE",
    ).sum()
)

csf_ratio_available_count = int(
    normalise_boolean_indicator(
        csf_df["CSF_RATIO_AVAILABLE"],
        column_name="CSF_RATIO_AVAILABLE",
    ).sum()
)

print("\nCSF AVAILABILITY DETAIL")
print(f"CSF_AVAILABLE = True: {csf_available_count:,}")
print(
    "CSF_AVAILABLE = False:",
    f"{AUTHORITATIVE_PARTICIPANT_COUNT - csf_available_count:,}",
)
print(
    "CSF_RATIO_AVAILABLE = True:",
    f"{csf_ratio_available_count:,}",
)

if csf_available_count != 1225:
    raise ValueError(
        "CSF_AVAILABLE does not match the validated count of 1,225."
    )

if csf_ratio_available_count != 366:
    raise ValueError(
        "CSF_RATIO_AVAILABLE does not match the validated count of 366."
    )

print(
    "\nCoverage and CSF ratio availability were validated successfully."
)

# 20. Inspect the columns of each modality before selecting predictors

The modality coverage checks have passed, so I can now examine the contents of each baseline-aligned manifest in more detail.

At this stage, I need to distinguish between:

- genuine clinical or biomarker predictors;
- participant identifiers;
- modality availability indicators;
- dates and baseline-alignment variables;
- quality-control fields;
- source and provenance fields;
- labels or outcome-related variables;
- file paths and image-processing metadata.

not automatically use every numeric column as a model feature. A numeric variable may still represent a date offset, source-row number, quality-control flag, image identifier, or another administrative field.

The code below creates a readable column inventory for each modality. It assigns every column a provisional role based on its name and prints the complete column list without shortening long names.

These provisional roles are only an organisational aid. Final feature selection will still be made explicitly for each modality.

In [ ]:
# ---------------------------------------------------------------------
# I create a readable modality-by-modality column inventory.
#
# Each column receives a provisional role based on its name. This helps
# me separate likely predictors from identifiers, timing variables,
# quality-control fields, file paths, availability indicators, and
# target-related information.
#
# This cell does not remove columns and does not yet define the final
# feature sets.
# ---------------------------------------------------------------------

def assign_provisional_column_role(
    column_name: str,
) -> str:
    """
    I assign a provisional role using the column name only.

    The order of these checks matters. More specific categories are
    checked before the broad candidate-feature category.
    """
    name = column_name.upper()

    # -------------------------------------------------------------
    # Participant identifiers
    # -------------------------------------------------------------
    if name in {
        "RID",
        "PTID",
    }:
        return "01_IDENTIFIER"

    # -------------------------------------------------------------
    # Availability and feature-presence indicators
    # -------------------------------------------------------------
    if any(
        token in name
        for token in [
            "AVAILABLE",
            "AVAILABILITY",
            "BIOMARKER_COUNT",
        ]
    ):
        return "02_AVAILABILITY"

    # -------------------------------------------------------------
    # Clinical labels and outcome-related fields
    #
    # These must be kept separate from predictors to avoid leakage.
    # -------------------------------------------------------------
    if any(
        token in name
        for token in [
            "GROUP",
            "CLASS",
            "DIAGNOSIS",
            "DIAG",
            "OUTCOME",
            "CONVERSION",
            "CONVERTER",
            "TRAJECTORY",
            "EXCLUSION",
            "TARGET",
        ]
    ):
        return "03_LABEL_OR_TARGET_RELATED"

    # -------------------------------------------------------------
    # Dates and baseline-alignment timing variables
    # -------------------------------------------------------------
    if any(
        token in name
        for token in [
            "DATE",
            "VISDATE",
            "EXAMDATE",
            "TESTDT",
            "DAYS_FROM_BASELINE",
            "ABS_DAYS_FROM_BASELINE",
        ]
    ):
        return "04_DATE_OR_TIMING"

    # -------------------------------------------------------------
    # File paths and image-processing provenance
    # -------------------------------------------------------------
    if any(
        token in name
        for token in [
            "PATH",
            "FILE",
            "NIFTI",
            "NPY",
            "DCM",
            "IMAGE_ID",
            "IMAGE_FOUND",
        ]
    ):
        return "05_FILE_OR_IMAGE_PROVENANCE"

    # -------------------------------------------------------------
    # Quality-control and validity fields
    # -------------------------------------------------------------
    if any(
        token in name
        for token in [
            "QC",
            "VALID",
            "USABLE",
            "EXISTS",
            "ERROR",
            "OUTLIER",
            "CONFLICT",
            "MISMATCH",
        ]
    ):
        return "06_QUALITY_CONTROL"

    # -------------------------------------------------------------
    # Administrative and source-tracking variables
    # -------------------------------------------------------------
    if any(
        token in name
        for token in [
            "PHASE",
            "VISCODE",
            "SOURCE",
            "ROW_INDEX",
            "SOURCE_ROW",
            "UPDATE_STAMP",
            "STAMP",
            "SITEID",
            "ID",
        ]
    ):
        return "07_ADMINISTRATIVE_OR_SOURCE"

    # -------------------------------------------------------------
    # Human-readable label versions of categorical variables
    #
    # These may duplicate coded variables and should be reviewed before
    # choosing which representation enters the model.
    # -------------------------------------------------------------
    if name.endswith("_LABEL"):
        return "08_HUMAN_READABLE_LABEL"

    # -------------------------------------------------------------
    # Anything not caught above remains a possible model feature.
    #
    # This does not mean that the column is automatically accepted.
    # It only means that it requires closer manual review.
    # -------------------------------------------------------------
    return "09_CANDIDATE_OR_UNCATEGORISED"


column_inventory_rows = []

for manifest_name, df in sorted(
    baseline_manifest_dfs.items()
):
    for column_position, column in enumerate(df.columns):
        series = df[column]
        non_missing_values = series.dropna()

        # I retain a short example value for orientation.
        if non_missing_values.empty:
            example_value = pd.NA
        else:
            example_value = non_missing_values.iloc[0]

            if isinstance(example_value, str):
                example_value = example_value.strip()

        column_inventory_rows.append(
            {
                "MANIFEST": manifest_name,
                "COLUMN_POSITION": column_position,
                "COLUMN": column,
                "PROVISIONAL_ROLE": (
                    assign_provisional_column_role(column)
                ),
                "DTYPE": str(series.dtype),
                "NON_MISSING": int(series.notna().sum()),
                "MISSING": int(series.isna().sum()),
                "MISSING_PERCENT": round(
                    100 * series.isna().mean(),
                    2,
                ),
                "UNIQUE_VALUES": int(
                    series.nunique(dropna=True)
                ),
                "EXAMPLE": example_value,
            }
        )

column_inventory_df = pd.DataFrame(
    column_inventory_rows
)

# ---------------------------------------------------------------------
# I print each manifest separately so that no modality is hidden inside
# one very large table.
# ---------------------------------------------------------------------
for manifest_name in sorted(
    column_inventory_df["MANIFEST"].unique()
):
    current_manifest = column_inventory_df.loc[
        column_inventory_df["MANIFEST"] == manifest_name
    ].copy()

    current_manifest = current_manifest.sort_values(
        [
            "PROVISIONAL_ROLE",
            "COLUMN_POSITION",
        ]
    )

    print("\n" + "=" * 120)
    print(f"MANIFEST: {manifest_name}")
    print("=" * 120)

    print(
        f"Rows: "
        f"{len(baseline_manifest_dfs[manifest_name]):,}"
    )

    print(
        f"Columns: "
        f"{baseline_manifest_dfs[manifest_name].shape[1]:,}"
    )

    if "RID" in baseline_manifest_dfs[manifest_name].columns:
        print(
            f"Unique RID: "
            f"{baseline_manifest_dfs[manifest_name]['RID'].nunique(dropna=True):,}"
        )

    print()

    # I use to_string so Colab prints every row and every column name
    # without hiding the middle of the table.
    print(
        current_manifest[
            [
                "COLUMN_POSITION",
                "COLUMN",
                "PROVISIONAL_ROLE",
                "DTYPE",
                "NON_MISSING",
                "MISSING_PERCENT",
                "UNIQUE_VALUES",
                "EXAMPLE",
            ]
        ].to_string(
            index=False,
            max_colwidth=120,
        )
    )

# ---------------------------------------------------------------------
# I save the complete structured inventory for later feature selection.
# ---------------------------------------------------------------------
COLUMN_INVENTORY_PATH = (
    MANIFEST_OUTPUT_DIR
    / "baseline_aligned_column_inventory.csv"
)

column_inventory_df.to_csv(
    COLUMN_INVENTORY_PATH,
    index=False,
)

if not COLUMN_INVENTORY_PATH.is_file():
    raise FileNotFoundError(
        "The baseline-aligned column inventory was not saved:\n"
        f"{COLUMN_INVENTORY_PATH}"
    )

print("\n" + "=" * 120)
print("Column inventory completed successfully.")
print(f"Saved: {COLUMN_INVENTORY_PATH}")

# 21. Define explicit candidate feature sets for each modelling modality

The column inventory confirms that the baseline-aligned manifests contain both genuine clinical measurements and many fields that exist only for audit, provenance, timing, or quality control.

therefore use explicit feature allowlists rather than selecting columns automatically by data type.

For the first modelling configuration, organise the non-imaging data into five branches:

1. **Demographics**
   - age;
   - sex;
   - education;
   - handedness.

2. **Cognitive and functional assessment**
   - ADAS total scores;
   - MMSE total and domain scores;
   - FAQ total score.

3. **CSF biomarkers**
   - Aβ40;
   - Aβ42;
   - total tau;
   - phosphorylated tau;
   - Aβ42/Aβ40 ratio.

4. **Plasma biomarkers**
   - plasma p-tau217;
   - Aβ42;
   - Aβ40;
   - Aβ42/Aβ40 ratio;
   - p-tau217/Aβ42 ratio;
   - available NfL and GFAP measurements.

5. **APOE genotype**
   - APOE ε4 allele count.

MRform a separate imaging branch using the final normalised T1 volume.

not include the following as predictors:

- clinical group labels;
- baseline diagnosis;
- dates or days from baseline;
- visit codes;
- ADNI phase;
- file paths other than the MRI loading path;
- quality-control flags;
- availability indicators as ordinary clinical measurements;
- scanner or preprocessing metadata;
- human-readable duplicate labels;
- biomarker batch or administrative fields.

The availability indicators will be retained separately because the model needs to know which modalities and features are present. They will not be mixed into the clinical feature vectors as if they were biological measurements.

This cell defines and validates the initial candidate feature configuration. It does not yet transform values, fill missing measurements, merge modalities, or create data splits.

In [ ]:
# ---------------------------------------------------------------------
# I define every candidate model feature explicitly.
#
# I do not select all numeric columns automatically. This protects the
# model from accidentally receiving dates, timing offsets, labels,
# quality-control variables, source identifiers, or processing metadata.
# ---------------------------------------------------------------------

MANIFEST_KEYS = {
    "adas": "adas__adas_baseline_aligned_cohort",
    "apoe": "apoe__apoe_baseline_aligned_cohort",
    "csf": (
        "csf_core_biomarkers__"
        "csf_core_biomarkers_baseline_aligned_cohort"
    ),
    "faq": "faq__faq_baseline_aligned_cohort",
    "mmse": "mmse__mmse_baseline_aligned_cohort",
    "mri": "mri__mri_baseline_aligned_authoritative_cohort_1057",
    "plasma": "plasma__plasma_baseline_aligned_cohort",
    "ptdemog": "ptdemog__ptdemog_baseline_aligned_cohort",
}

# ---------------------------------------------------------------------
# Demographic features
#
# I begin with age, sex, education, and handedness.
#
# I do not include race, ethnicity, language, or marital status in the
# initial model. Those variables require separate scientific and fairness
# justification and may encode recruitment-site or socioeconomic effects
# that are not the biological target of this thesis.
# ---------------------------------------------------------------------
DEMOGRAPHIC_FEATURES = [
    "AGE_AT_BASELINE",
    "PTGENDER_CLEAN",
    "PTEDUCAT_CLEAN",
    "PTHAND_CLEAN",
]

# ---------------------------------------------------------------------
# ADAS features
#
# TOTSCORE and TOTAL13 represent related but distinct ADAS summaries.
# I retain both as initial candidates and will later inspect redundancy
# and decide whether both should enter the final cognitive encoder.
# ---------------------------------------------------------------------
ADAS_FEATURES = [
    "TOTSCORE",
    "TOTAL13",
]

# ---------------------------------------------------------------------
# MMSE features
#
# I retain the total score and the cleaned domain-level scores.
#
# I exclude:
#   - record keys;
#   - visit codes;
#   - remote-assessment flags;
#   - consistency and mismatch flags;
#   - quality-control fields.
# ---------------------------------------------------------------------
MMSE_FEATURES = [
    "MMSE_TOTAL_SCORE",
    "MMSE_ORIENTATION_SCORE",
    "MMSE_REGISTRATION_SCORE",
    "MMSE_ATTENTION_SCORE",
    "MMSE_DELAYED_RECALL_SCORE",
    "MMSE_LANGUAGE_COMMAND_SCORE",
]

# ---------------------------------------------------------------------
# FAQ feature
# ---------------------------------------------------------------------
FAQ_FEATURES = [
    "FAQTOTAL",
]

# ---------------------------------------------------------------------
# CSF biomarker features
#
# Feature-specific missingness is expected. For example, a participant
# may have ABETA42, TAU, or PTAU while lacking ABETA40 and therefore
# lacking the ABETA42/40 ratio.
# ---------------------------------------------------------------------
CSF_FEATURES = [
    "ABETA40",
    "ABETA42",
    "TAU",
    "PTAU",
    "ABETA42_40_RATIO",
]

# ---------------------------------------------------------------------
# Plasma biomarker features
#
# I retain the available measurements from both assay families. Their
# feature-level missingness will be represented explicitly later.
# ---------------------------------------------------------------------
PLASMA_FEATURES = [
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
    "NfL_Q",
    "GFAP_Q",
    "NfL_F",
    "GFAP_F",
]

# ---------------------------------------------------------------------
# APOE feature
#
# APOE4_ALLELE_COUNT contains the clinically interpretable ε4 dosage:
#   0 = no ε4 allele;
#   1 = one ε4 allele;
#   2 = two ε4 alleles.
#
# I do not simultaneously include allele 1, allele 2, genotype string,
# carrier status, and allele count because they encode overlapping
# information.
# ---------------------------------------------------------------------
APOE_FEATURES = [
    "APOE4_ALLELE_COUNT",
]

# ---------------------------------------------------------------------
# MRI model input
#
# The MRI branch uses the complete normalised three-dimensional T1
# volume, not scalar scanner or preprocessing-summary variables.
# ---------------------------------------------------------------------
MRI_PATH_COLUMN = "normalized_t1_npy_path"

# ---------------------------------------------------------------------
# Overall modality-level availability indicators
#
# MRI availability is determined by membership in the authoritative
# 1,057-row MRI manifest rather than by a stored Boolean column.
# ---------------------------------------------------------------------
MODALITY_AVAILABILITY_COLUMNS = {
    "demographics": None,
    "adas": "ADAS_AVAILABLE",
    "mmse": "MMSE_AVAILABLE",
    "faq": "FAQ_AVAILABLE",
    "csf": "CSF_AVAILABLE",
    "plasma": "PLASMA_AVAILABLE",
    "apoe": "APOE_AVAILABLE",
    "mri": None,
}

# ---------------------------------------------------------------------
# Feature-level availability indicators that should be retained for
# later missingness-mask construction.
#
# These are not ordinary predictor values. They describe which subsets
# of a modality are genuinely observed.
# ---------------------------------------------------------------------
FEATURE_LEVEL_AVAILABILITY_COLUMNS = {
    "csf": [
        "CSF_RATIO_AVAILABLE",
        "ABETA42_40_RATIO_AVAILABLE",
        "AVAILABLE_CORE_BIOMARKER_COUNT",
    ],
    "plasma": [
        "PLASMA_AMYLOID_PTAU_AVAILABLE",
        "PLASMA_QUANTERIX_NFL_GFAP_AVAILABLE",
        "PLASMA_FUJIREBIO_NFL_GFAP_AVAILABLE",
        "AVAILABLE_PLASMA_BIOMARKER_COUNT",
    ],
}

# ---------------------------------------------------------------------
# I collect the configuration into one dictionary so that later cells
# can access it consistently.
# ---------------------------------------------------------------------
CANDIDATE_FEATURES = {
    "demographics": {
        "manifest_key": MANIFEST_KEYS["ptdemog"],
        "features": DEMOGRAPHIC_FEATURES,
    },
    "adas": {
        "manifest_key": MANIFEST_KEYS["adas"],
        "features": ADAS_FEATURES,
    },
    "mmse": {
        "manifest_key": MANIFEST_KEYS["mmse"],
        "features": MMSE_FEATURES,
    },
    "faq": {
        "manifest_key": MANIFEST_KEYS["faq"],
        "features": FAQ_FEATURES,
    },
    "csf": {
        "manifest_key": MANIFEST_KEYS["csf"],
        "features": CSF_FEATURES,
    },
    "plasma": {
        "manifest_key": MANIFEST_KEYS["plasma"],
        "features": PLASMA_FEATURES,
    },
    "apoe": {
        "manifest_key": MANIFEST_KEYS["apoe"],
        "features": APOE_FEATURES,
    },
    "mri": {
        "manifest_key": MANIFEST_KEYS["mri"],
        "features": [MRI_PATH_COLUMN],
    },
}

# ---------------------------------------------------------------------
# I validate every manifest key, candidate feature, and availability
# indicator before proceeding.
#
# The notebook should stop immediately if a configured column is absent.
# A misspelled or outdated feature name must never be silently ignored.
# ---------------------------------------------------------------------
configuration_validation_rows = []

for modality, configuration in CANDIDATE_FEATURES.items():
    manifest_key = configuration["manifest_key"]

    if manifest_key not in baseline_manifest_dfs:
        raise KeyError(
            f"The configured manifest for {modality!r} was not loaded:\n"
            f"{manifest_key}"
        )

    manifest_df = baseline_manifest_dfs[manifest_key]

    for feature in configuration["features"]:
        exists = feature in manifest_df.columns

        configuration_validation_rows.append(
            {
                "MODALITY": modality.upper(),
                "COLUMN_ROLE": "candidate_feature",
                "COLUMN": feature,
                "MANIFEST": manifest_key,
                "EXISTS": exists,
                "DTYPE": (
                    str(manifest_df[feature].dtype)
                    if exists
                    else pd.NA
                ),
                "NON_MISSING": (
                    int(manifest_df[feature].notna().sum())
                    if exists
                    else pd.NA
                ),
                "MISSING_PERCENT": (
                    round(
                        100 * manifest_df[feature].isna().mean(),
                        2,
                    )
                    if exists
                    else pd.NA
                ),
            }
        )

    availability_column = MODALITY_AVAILABILITY_COLUMNS.get(
        modality
    )

    if availability_column is not None:
        exists = availability_column in manifest_df.columns

        configuration_validation_rows.append(
            {
                "MODALITY": modality.upper(),
                "COLUMN_ROLE": "modality_availability",
                "COLUMN": availability_column,
                "MANIFEST": manifest_key,
                "EXISTS": exists,
                "DTYPE": (
                    str(manifest_df[availability_column].dtype)
                    if exists
                    else pd.NA
                ),
                "NON_MISSING": (
                    int(
                        manifest_df[
                            availability_column
                        ].notna().sum()
                    )
                    if exists
                    else pd.NA
                ),
                "MISSING_PERCENT": (
                    round(
                        100
                        * manifest_df[
                            availability_column
                        ].isna().mean(),
                        2,
                    )
                    if exists
                    else pd.NA
                ),
            }
        )

for modality, availability_columns in (
    FEATURE_LEVEL_AVAILABILITY_COLUMNS.items()
):
    manifest_key = CANDIDATE_FEATURES[modality][
        "manifest_key"
    ]

    manifest_df = baseline_manifest_dfs[manifest_key]

    for column in availability_columns:
        exists = column in manifest_df.columns

        configuration_validation_rows.append(
            {
                "MODALITY": modality.upper(),
                "COLUMN_ROLE": "feature_level_availability",
                "COLUMN": column,
                "MANIFEST": manifest_key,
                "EXISTS": exists,
                "DTYPE": (
                    str(manifest_df[column].dtype)
                    if exists
                    else pd.NA
                ),
                "NON_MISSING": (
                    int(manifest_df[column].notna().sum())
                    if exists
                    else pd.NA
                ),
                "MISSING_PERCENT": (
                    round(
                        100 * manifest_df[column].isna().mean(),
                        2,
                    )
                    if exists
                    else pd.NA
                ),
            }
        )

feature_configuration_validation_df = pd.DataFrame(
    configuration_validation_rows
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    250,
):
    display(feature_configuration_validation_df)

missing_configured_columns = (
    feature_configuration_validation_df.loc[
        ~feature_configuration_validation_df["EXISTS"]
    ]
)

if not missing_configured_columns.empty:
    display(missing_configured_columns)

    raise KeyError(
        "At least one explicitly configured feature or availability "
        "column is absent from its expected manifest."
    )

# ---------------------------------------------------------------------
# I print a concise feature count by modality.
# ---------------------------------------------------------------------
feature_count_summary = pd.DataFrame(
    [
        {
            "MODALITY": modality.upper(),
            "CANDIDATE_FEATURE_COUNT": len(
                configuration["features"]
            ),
            "CANDIDATE_FEATURES": ", ".join(
                configuration["features"]
            ),
        }
        for modality, configuration in (
            CANDIDATE_FEATURES.items()
        )
    ]
)

with pd.option_context(
    "display.max_colwidth",
    None,
    "display.width",
    250,
):
    display(feature_count_summary)

print(
    "All explicitly configured candidate features and availability "
    "columns were found in their expected manifests."
)

# 22. Create the authoritative participant base using the known cohort columns

I created the authoritative cohort file myself, so use its established columns directly rather than rediscovering their meaning.

The source columns are:

- `COHORT_LABEL`: the authoritative four-group clinical label;
- `TRAJECTORY_LABEL`: the MCI prognosis label, populated only for pMCI and sMCI participants;
- `PROGNOSIS_TARGET`: the binary MCI prognosis target;
- `RID`: the participant identifier;
- `PTID`: the ADNI participant identifier.

The modelling base will retain all 2,199 authoritative participants.

preserve both the four-group label and the MCI prognosis target because they support different experiments:

- four-group classification uses CN, AD, sMCI, and pMCI;
- prognosis modelling uses only participants with an MCI trajectory label;
- `PROGNOSIS_TARGET` will later be checked before it is used as the binary target.

This cell does not merge any modalities yet. It creates the clean participant-level foundation onto which demographics, cognitive assessments, biomarkers, APOE, and MRI availability will later be added.

In [ ]:
# ---------------------------------------------------------------------
# I use the known authoritative cohort columns directly.
#
# I do not repeat automatic label discovery because these columns were
# deliberately created during cohort construction and have already been
# validated.
# ---------------------------------------------------------------------

AUTHORITATIVE_LABEL_COLUMN = "COHORT_LABEL"
MCI_TRAJECTORY_COLUMN = "TRAJECTORY_LABEL"
MCI_PROGNOSIS_TARGET_COLUMN = "PROGNOSIS_TARGET"

EXPECTED_GROUP_COUNTS = {
    "CN": 1196,
    "AD": 459,
    "sMCI": 295,
    "pMCI": 249,
}

REQUIRED_AUTHORITATIVE_COLUMNS = [
    "RID",
    "PTID",
    AUTHORITATIVE_LABEL_COLUMN,
    MCI_TRAJECTORY_COLUMN,
    MCI_PROGNOSIS_TARGET_COLUMN,
]

# ---------------------------------------------------------------------
# I confirm that the expected authoritative columns are present.
#
# This is a structural check only. I am not trying to infer or recreate
# the labels.
# ---------------------------------------------------------------------
missing_authoritative_columns = [
    column
    for column in REQUIRED_AUTHORITATIVE_COLUMNS
    if column not in cohort_df.columns
]

if missing_authoritative_columns:
    raise KeyError(
        "The authoritative cohort is missing required columns: "
        + ", ".join(missing_authoritative_columns)
    )

# ---------------------------------------------------------------------
# I create one participant-level modelling base.
#
# I rename COHORT_LABEL to CLINICAL_GROUP only inside the modelling
# table so that later code has a clear and consistent target name.
#
# The source cohort dataframe remains unchanged.
# ---------------------------------------------------------------------
authoritative_model_base_df = (
    cohort_df[
        REQUIRED_AUTHORITATIVE_COLUMNS
    ]
    .copy()
    .rename(
        columns={
            AUTHORITATIVE_LABEL_COLUMN: "CLINICAL_GROUP",
            MCI_TRAJECTORY_COLUMN: "MCI_TRAJECTORY_LABEL",
            MCI_PROGNOSIS_TARGET_COLUMN: "MCI_PROGNOSIS_TARGET",
        }
    )
)

# ---------------------------------------------------------------------
# I standardise only basic storage types.
#
# I do not alter the meaning of any target values.
# ---------------------------------------------------------------------
authoritative_model_base_df["RID"] = (
    pd.to_numeric(
        authoritative_model_base_df["RID"],
        errors="coerce",
    )
    .astype("Int64")
)

authoritative_model_base_df["PTID"] = (
    authoritative_model_base_df["PTID"]
    .astype("string")
    .str.strip()
)

authoritative_model_base_df["CLINICAL_GROUP"] = (
    authoritative_model_base_df["CLINICAL_GROUP"]
    .astype("string")
    .str.strip()
)

authoritative_model_base_df["MCI_TRAJECTORY_LABEL"] = (
    authoritative_model_base_df["MCI_TRAJECTORY_LABEL"]
    .astype("string")
    .str.strip()
)

authoritative_model_base_df["MCI_PROGNOSIS_TARGET"] = (
    pd.to_numeric(
        authoritative_model_base_df["MCI_PROGNOSIS_TARGET"],
        errors="coerce",
    )
    .astype("Int64")
)

# ---------------------------------------------------------------------
# I validate the participant-level structure.
# ---------------------------------------------------------------------
if len(authoritative_model_base_df) != 2199:
    raise ValueError(
        "The authoritative modelling base should contain 2,199 rows, "
        f"but contains {len(authoritative_model_base_df):,}."
    )

if authoritative_model_base_df["RID"].isna().any():
    raise ValueError(
        "The authoritative modelling base contains missing RIDs."
    )

if authoritative_model_base_df["RID"].duplicated().any():
    duplicated_rids = authoritative_model_base_df.loc[
        authoritative_model_base_df["RID"].duplicated(
            keep=False
        ),
        ["RID", "PTID"],
    ]

    display(duplicated_rids)

    raise ValueError(
        "The authoritative modelling base contains duplicated RIDs."
    )

if authoritative_model_base_df["PTID"].isna().any():
    raise ValueError(
        "The authoritative modelling base contains missing PTIDs."
    )

# ---------------------------------------------------------------------
# I verify the known four-group counts.
#
# This does not rediscover the label. It simply confirms that the loaded
# file still matches the authoritative cohort that was previously built.
# ---------------------------------------------------------------------
observed_group_counts = (
    authoritative_model_base_df["CLINICAL_GROUP"]
    .value_counts(dropna=False)
    .reindex(
        ["CN", "AD", "sMCI", "pMCI"],
        fill_value=0,
    )
)

group_count_validation_df = pd.DataFrame(
    [
        {
            "CLINICAL_GROUP": group,
            "EXPECTED_COUNT": expected_count,
            "OBSERVED_COUNT": int(
                observed_group_counts[group]
            ),
            "MATCHES_EXPECTATION": (
                int(observed_group_counts[group])
                == expected_count
            ),
        }
        for group, expected_count in (
            EXPECTED_GROUP_COUNTS.items()
        )
    ]
)

display(group_count_validation_df)

if not group_count_validation_df[
    "MATCHES_EXPECTATION"
].all():
    raise ValueError(
        "The loaded four-group counts do not match the established "
        "authoritative cohort counts."
    )

# ---------------------------------------------------------------------
# I inspect the MCI prognosis subset without changing it.
#
# Only sMCI and pMCI participants should have an MCI trajectory label.
# The expected prognosis cohort contains:
#   - 295 sMCI participants;
#   - 249 pMCI participants;
#   - 544 participants in total.
# ---------------------------------------------------------------------
mci_prognosis_base_df = (
    authoritative_model_base_df.loc[
        authoritative_model_base_df[
            "CLINICAL_GROUP"
        ].isin(["sMCI", "pMCI"])
    ]
    .copy()
    .reset_index(drop=True)
)

expected_mci_count = (
    EXPECTED_GROUP_COUNTS["sMCI"]
    + EXPECTED_GROUP_COUNTS["pMCI"]
)

if len(mci_prognosis_base_df) != expected_mci_count:
    raise ValueError(
        "The MCI prognosis subset should contain "
        f"{expected_mci_count:,} participants, but contains "
        f"{len(mci_prognosis_base_df):,}."
    )

if mci_prognosis_base_df[
    "MCI_TRAJECTORY_LABEL"
].isna().any():
    missing_trajectory_rows = (
        mci_prognosis_base_df.loc[
            mci_prognosis_base_df[
                "MCI_TRAJECTORY_LABEL"
            ].isna()
        ]
    )

    display(missing_trajectory_rows)

    raise ValueError(
        "At least one sMCI or pMCI participant is missing the "
        "MCI trajectory label."
    )

# ---------------------------------------------------------------------
# I confirm that non-MCI participants do not unexpectedly contain an
# MCI trajectory label.
# ---------------------------------------------------------------------
non_mci_rows = authoritative_model_base_df.loc[
    ~authoritative_model_base_df[
        "CLINICAL_GROUP"
    ].isin(["sMCI", "pMCI"])
]

unexpected_non_mci_trajectory_rows = non_mci_rows.loc[
    non_mci_rows["MCI_TRAJECTORY_LABEL"].notna()
]

if not unexpected_non_mci_trajectory_rows.empty:
    display(
        unexpected_non_mci_trajectory_rows.head(30)
    )

    raise ValueError(
        "At least one CN or AD participant unexpectedly contains an "
        "MCI trajectory label."
    )

# ---------------------------------------------------------------------
# I display compact summaries of the full cohort and prognosis subset.
# ---------------------------------------------------------------------
print(
    "Authoritative modelling base:",
    f"{len(authoritative_model_base_df):,} participants",
)

print(
    "MCI prognosis base:",
    f"{len(mci_prognosis_base_df):,} participants",
)

print("\nMCI trajectory distribution:")

display(
    mci_prognosis_base_df[
        "MCI_TRAJECTORY_LABEL"
    ]
    .value_counts(dropna=False)
    .rename_axis("MCI_TRAJECTORY_LABEL")
    .reset_index(name="COUNT")
)

print("\nExample authoritative participant rows:")

display(
    authoritative_model_base_df.head(10)
)

# 23. Build the participant-level multimodal modelling table

combine the authoritative participant base with the explicitly selected features from every baseline-aligned modality.

The authoritative cohort remains the left-hand table. This means that all 2,199 participants are retained even when one or more modalities are unavailable.

use the following merge rules:

- merge by `RID`;
- require each modality table to contain at most one row per RID;
- retain only the explicitly selected features and availability fields;
- add a modality prefix to every merged column;
- preserve missing feature values exactly as missing;
- create one explicit Boolean availability indicator for each modality;
- add the validated normalised T1 MRI path only for the 1,057 MRI participants.

The resulting table will be the unprocessed participant-level multimodal source table. Leakage-free transformations will be fitted only after the data splits are created.

In [ ]:
# ---------------------------------------------------------------------
# I build one participant-level multimodal table by left-merging every
# selected modality onto the authoritative 2,199-participant base.
#
# The authoritative cohort is always the left-hand table. Therefore,
# missing modalities remain missing instead of causing participant
# removal.
# ---------------------------------------------------------------------

multimodal_master_df = authoritative_model_base_df.copy()

# ---------------------------------------------------------------------
# I start by creating an explicit demographics availability indicator.
#
# PTDEMOG contains all 2,199 authoritative participants. However, an
# individual demographic feature can still be missing. Therefore:
#
#   DEMOGRAPHICS_AVAILABLE = True
#
# means that the participant is represented in the demographics
# manifest. It does not claim that every demographic feature is present.
# Feature-level missingness will be represented separately later.
# ---------------------------------------------------------------------
ptdemog_key = CANDIDATE_FEATURES["demographics"]["manifest_key"]
ptdemog_source_df = baseline_manifest_dfs[ptdemog_key]

demographic_columns_to_merge = (
    ["RID"]
    + CANDIDATE_FEATURES["demographics"]["features"]
)

demographics_merge_df = (
    ptdemog_source_df[
        demographic_columns_to_merge
    ]
    .copy()
)

demographics_merge_df["DEMOGRAPHICS_AVAILABLE"] = True

# I prefix the selected demographic feature names so that the source
# modality remains clear after all tables are combined.
demographic_rename_map = {
    feature: f"DEMOGRAPHICS__{feature}"
    for feature in CANDIDATE_FEATURES[
        "demographics"
    ]["features"]
}

demographics_merge_df = demographics_merge_df.rename(
    columns=demographic_rename_map
)

multimodal_master_df = multimodal_master_df.merge(
    demographics_merge_df,
    on="RID",
    how="left",
    validate="one_to_one",
)

# Every authoritative participant should be represented in PTDEMOG.
if multimodal_master_df[
    "DEMOGRAPHICS_AVAILABLE"
].isna().any():
    missing_demographic_rows = multimodal_master_df.loc[
        multimodal_master_df[
            "DEMOGRAPHICS_AVAILABLE"
        ].isna(),
        ["RID", "PTID"],
    ]

    display(missing_demographic_rows.head(30))

    raise ValueError(
        "At least one authoritative participant was not matched to the "
        "PTDEMOG baseline-aligned manifest."
    )

multimodal_master_df["DEMOGRAPHICS_AVAILABLE"] = (
    multimodal_master_df[
        "DEMOGRAPHICS_AVAILABLE"
    ].astype(bool)
)


# ---------------------------------------------------------------------
# I define a reusable function for participant-level non-imaging
# modalities that already contain one row for every authoritative RID.
# ---------------------------------------------------------------------
def merge_selected_non_imaging_modality(
    master_df: pd.DataFrame,
    modality_name: str,
) -> pd.DataFrame:
    """
    I merge one selected non-imaging modality onto the authoritative
    participant table.

    The function:
      1. loads the configured manifest from memory;
      2. selects only RID, configured features, the overall availability
         indicator, and configured feature-level availability fields;
      3. prefixes merged columns with the modality name;
      4. performs a validated one-to-one left merge;
      5. confirms that the overall availability indicator survives the
         merge without missing participant rows.

    I do not fill missing clinical or biomarker measurements.
    """
    configuration = CANDIDATE_FEATURES[modality_name]
    manifest_key = configuration["manifest_key"]
    feature_columns = configuration["features"]

    source_df = baseline_manifest_dfs[
        manifest_key
    ]

    overall_availability_column = (
        MODALITY_AVAILABILITY_COLUMNS[
            modality_name
        ]
    )

    feature_level_availability_columns = (
        FEATURE_LEVEL_AVAILABILITY_COLUMNS.get(
            modality_name,
            [],
        )
    )

    columns_to_merge = (
        ["RID"]
        + feature_columns
        + [overall_availability_column]
        + feature_level_availability_columns
    )

    # I preserve column order while removing accidental duplicates.
    columns_to_merge = list(
        dict.fromkeys(columns_to_merge)
    )

    missing_columns = [
        column
        for column in columns_to_merge
        if column not in source_df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"The {modality_name} merge configuration contains columns "
            f"that are absent from {manifest_key}: "
            + ", ".join(missing_columns)
        )

    modality_merge_df = source_df[
        columns_to_merge
    ].copy()

    if modality_merge_df["RID"].isna().any():
        raise ValueError(
            f"The {modality_name} merge table contains missing RIDs."
        )

    if modality_merge_df["RID"].duplicated().any():
        duplicated_rids = modality_merge_df.loc[
            modality_merge_df["RID"].duplicated(
                keep=False
            ),
            ["RID"],
        ]

        display(duplicated_rids.head(30))

        raise ValueError(
            f"The {modality_name} merge table contains duplicated RIDs."
        )

    prefix = modality_name.upper()

    rename_map = {
        column: f"{prefix}__{column}"
        for column in columns_to_merge
        if column != "RID"
    }

    modality_merge_df = modality_merge_df.rename(
        columns=rename_map
    )

    rows_before_merge = len(master_df)

    merged_df = master_df.merge(
        modality_merge_df,
        on="RID",
        how="left",
        validate="one_to_one",
    )

    if len(merged_df) != rows_before_merge:
        raise ValueError(
            f"Merging {modality_name} changed the authoritative "
            f"participant count from {rows_before_merge:,} to "
            f"{len(merged_df):,}."
        )

    prefixed_availability_column = (
        f"{prefix}__{overall_availability_column}"
    )

    # Because these manifests contain all 2,199 authoritative rows, the
    # overall availability flag itself should never become missing after
    # the merge. False is a valid value; missing is not.
    if merged_df[
        prefixed_availability_column
    ].isna().any():
        unmatched_rows = merged_df.loc[
            merged_df[
                prefixed_availability_column
            ].isna(),
            ["RID", "PTID"],
        ]

        display(unmatched_rows.head(30))

        raise ValueError(
            f"The {modality_name} manifest did not match every "
            "authoritative participant."
        )

    merged_df[
        prefixed_availability_column
    ] = normalise_boolean_indicator(
        merged_df[
            prefixed_availability_column
        ],
        column_name=prefixed_availability_column,
    )

    return merged_df


# ---------------------------------------------------------------------
# I merge each selected non-imaging modality.
#
# These manifests contain all 2,199 authoritative participant rows and
# use their overall availability indicators to distinguish observed from
# unavailable modalities.
# ---------------------------------------------------------------------
for modality_name in [
    "adas",
    "mmse",
    "faq",
    "csf",
    "plasma",
    "apoe",
]:
    multimodal_master_df = (
        merge_selected_non_imaging_modality(
            multimodal_master_df,
            modality_name,
        )
    )

    print(
        f"Merged {modality_name.upper():<10} | "
        f"rows={len(multimodal_master_df):,} | "
        f"columns={multimodal_master_df.shape[1]:,}"
    )


# ---------------------------------------------------------------------
# I merge the final normalised T1 MRI path.
#
# Unlike the non-imaging baseline-aligned manifests, the MRI manifest
# contains only the 1,057 participants with an accepted MRI.
#
# Therefore, MRI availability is determined by whether a participant
# receives a validated NPY path after the left merge.
# ---------------------------------------------------------------------
required_mri_merge_columns = [
    "RID",
    "NPY_PATH",
    "NPY_PATH_EXISTS",
]

missing_mri_merge_columns = [
    column
    for column in required_mri_merge_columns
    if column not in mri_model_df.columns
]

if missing_mri_merge_columns:
    raise KeyError(
        "The validated MRI model table is missing required columns: "
        + ", ".join(missing_mri_merge_columns)
    )

mri_merge_df = (
    mri_model_df[
        required_mri_merge_columns
    ]
    .copy()
    .rename(
        columns={
            "NPY_PATH": "MRI__NORMALIZED_T1_NPY_PATH",
            "NPY_PATH_EXISTS": "MRI__PATH_EXISTS",
        }
    )
)

if mri_merge_df["RID"].duplicated().any():
    raise ValueError(
        "The validated MRI merge table contains duplicated RIDs."
    )

if not mri_merge_df["MRI__PATH_EXISTS"].all():
    invalid_mri_paths = mri_merge_df.loc[
        ~mri_merge_df["MRI__PATH_EXISTS"]
    ]

    display(invalid_mri_paths.head(30))

    raise FileNotFoundError(
        "At least one MRI path marked for modelling does not exist."
    )

rows_before_mri_merge = len(
    multimodal_master_df
)

multimodal_master_df = multimodal_master_df.merge(
    mri_merge_df,
    on="RID",
    how="left",
    validate="one_to_one",
)

if len(multimodal_master_df) != rows_before_mri_merge:
    raise ValueError(
        "The MRI merge changed the authoritative participant count."
    )

multimodal_master_df["MRI_AVAILABLE"] = (
    multimodal_master_df[
        "MRI__NORMALIZED_T1_NPY_PATH"
    ].notna()
)

# MRI__PATH_EXISTS is an audit variable. For participants without MRI,
# it is missing after the left merge. I convert this audit field to a
# simple Boolean value for consistency.
multimodal_master_df["MRI__PATH_EXISTS"] = (
    multimodal_master_df[
        "MRI__PATH_EXISTS"
    ]
    .fillna(False)
    .astype(bool)
)


# ---------------------------------------------------------------------
# Final participant-level structural validation
# ---------------------------------------------------------------------
if len(multimodal_master_df) != 2199:
    raise ValueError(
        "The multimodal master table should contain 2,199 rows, "
        f"but contains {len(multimodal_master_df):,}."
    )

if multimodal_master_df["RID"].nunique() != 2199:
    raise ValueError(
        "The multimodal master table should contain 2,199 unique RIDs."
    )

if multimodal_master_df["RID"].duplicated().any():
    raise ValueError(
        "The multimodal master table contains duplicated RIDs."
    )

# ---------------------------------------------------------------------
# I collect the final overall availability columns.
# ---------------------------------------------------------------------
MASTER_AVAILABILITY_COLUMNS = [
    "DEMOGRAPHICS_AVAILABLE",
    "ADAS__ADAS_AVAILABLE",
    "MMSE__MMSE_AVAILABLE",
    "FAQ__FAQ_AVAILABLE",
    "CSF__CSF_AVAILABLE",
    "PLASMA__PLASMA_AVAILABLE",
    "APOE__APOE_AVAILABLE",
    "MRI_AVAILABLE",
]

missing_master_availability_columns = [
    column
    for column in MASTER_AVAILABILITY_COLUMNS
    if column not in multimodal_master_df.columns
]

if missing_master_availability_columns:
    raise KeyError(
        "The multimodal master table is missing availability columns: "
        + ", ".join(
            missing_master_availability_columns
        )
    )

# ---------------------------------------------------------------------
# I verify that the merged availability counts still match the counts
# already validated from the source manifests.
# ---------------------------------------------------------------------
merged_availability_expectations = {
    "DEMOGRAPHICS_AVAILABLE": 2199,
    "ADAS__ADAS_AVAILABLE": 2096,
    "MMSE__MMSE_AVAILABLE": 1828,
    "FAQ__FAQ_AVAILABLE": 2098,
    "CSF__CSF_AVAILABLE": 1225,
    "PLASMA__PLASMA_AVAILABLE": 841,
    "APOE__APOE_AVAILABLE": 2126,
    "MRI_AVAILABLE": 1057,
}

merged_coverage_validation_rows = []

for column, expected_count in (
    merged_availability_expectations.items()
):
    observed_count = int(
        multimodal_master_df[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    merged_coverage_validation_rows.append(
        {
            "AVAILABILITY_COLUMN": column,
            "EXPECTED_AVAILABLE": expected_count,
            "OBSERVED_AVAILABLE": observed_count,
            "MATCHES_EXPECTATION": (
                observed_count == expected_count
            ),
        }
    )

merged_coverage_validation_df = pd.DataFrame(
    merged_coverage_validation_rows
)

display(merged_coverage_validation_df)

if not merged_coverage_validation_df[
    "MATCHES_EXPECTATION"
].all():
    raise ValueError(
        "At least one modality availability count changed during the "
        "participant-level merge."
    )

print(
    "\nParticipant-level multimodal master table created successfully."
)

print(
    f"Rows: {len(multimodal_master_df):,}"
)

print(
    f"Columns: {multimodal_master_df.shape[1]:,}"
)

display(
    multimodal_master_df.head(10)
)

# 24. Define the final modality branches and feature roles

freeze the structure that the separate model-training notebook will receive.

The participant-level master table contains several original assessment tables, but the model does not need one separate encoder for every source CSV. Instead, organise the selected inputs into six clinically meaningful branches:

1. **Demographics**
   - age;
   - education;
   - sex;
   - handedness.

2. **Cognitive and functional assessment**
   - ADAS;
   - MMSE;
   - FAQ.

3. **CSF biomarkers**
   - Aβ40;
   - Aβ42;
   - total tau;
   - phosphorylated tau;
   - Aβ42/Aβ40 ratio.

4. **Plasma biomarkers**
   - p-tau217;
   - Aβ42;
   - Aβ40;
   - plasma ratios;
   - NfL;
   - GFAP.

5. **APOE genotype**
   - APOE ε4 allele count.

6. **MRI**
   - the final normalised three-dimensional T1 image path.

Within the demographic branch, age and education will be treated as continuous numerical variables. Sex and handedness will be treated as categorical variables.

APOE ε4 allele count will also be treated as categorical or ordinal information rather than being standardised like a continuous laboratory value.

ADAS, MMSE, FAQ, CSF, and plasma measurements will be treated as continuous numerical features and will later be standardised using training-set statistics only.

The cognitive and functional branch combines three source assessments, but their individual availability indicators will be preserved:

- `ADAS__ADAS_AVAILABLE`;
- `MMSE__MMSE_AVAILABLE`;
- `FAQ__FAQ_AVAILABLE`.

The overall cognitive-functional branch will be considered available when at least one of these three assessments is available.

This cell defines the final schema only. It does not yet create feature masks, scale values, encode categories, or split participants.

In [ ]:
# ---------------------------------------------------------------------
# I define the final modality branches that will be passed to the
# separate model-training notebook.
#
# This schema is explicit and fixed. I do not infer branches from column
# names automatically because the architecture should receive a stable,
# documented feature configuration.
# ---------------------------------------------------------------------

FINAL_BRANCH_SCHEMA = {
    "demographics": {
        "continuous_features": [
            "DEMOGRAPHICS__AGE_AT_BASELINE",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN",
        ],
        "categorical_features": [
            "DEMOGRAPHICS__PTGENDER_CLEAN",
            "DEMOGRAPHICS__PTHAND_CLEAN",
        ],
        "availability_columns": [
            "DEMOGRAPHICS_AVAILABLE",
        ],
    },

    "cognitive_functional": {
        "continuous_features": [
            # ADAS summary scores
            "ADAS__TOTSCORE",
            "ADAS__TOTAL13",

            # MMSE total and domain scores
            "MMSE__MMSE_TOTAL_SCORE",
            "MMSE__MMSE_ORIENTATION_SCORE",
            "MMSE__MMSE_REGISTRATION_SCORE",
            "MMSE__MMSE_ATTENTION_SCORE",
            "MMSE__MMSE_DELAYED_RECALL_SCORE",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE",

            # FAQ total score
            "FAQ__FAQTOTAL",
        ],
        "categorical_features": [],
        "availability_columns": [
            "ADAS__ADAS_AVAILABLE",
            "MMSE__MMSE_AVAILABLE",
            "FAQ__FAQ_AVAILABLE",
        ],
    },

    "csf": {
        "continuous_features": [
            "CSF__ABETA40",
            "CSF__ABETA42",
            "CSF__TAU",
            "CSF__PTAU",
            "CSF__ABETA42_40_RATIO",
        ],
        "categorical_features": [],
        "availability_columns": [
            "CSF__CSF_AVAILABLE",
        ],
        "feature_level_availability_columns": [
            "CSF__CSF_RATIO_AVAILABLE",
            "CSF__ABETA42_40_RATIO_AVAILABLE",
            "CSF__AVAILABLE_CORE_BIOMARKER_COUNT",
        ],
    },

    "plasma": {
        "continuous_features": [
            "PLASMA__pT217_F",
            "PLASMA__AB42_F",
            "PLASMA__AB40_F",
            "PLASMA__AB42_AB40_F",
            "PLASMA__pT217_AB42_F",
            "PLASMA__NfL_Q",
            "PLASMA__GFAP_Q",
            "PLASMA__NfL_F",
            "PLASMA__GFAP_F",
        ],
        "categorical_features": [],
        "availability_columns": [
            "PLASMA__PLASMA_AVAILABLE",
        ],
        "feature_level_availability_columns": [
            "PLASMA__PLASMA_AMYLOID_PTAU_AVAILABLE",
            "PLASMA__PLASMA_QUANTERIX_NFL_GFAP_AVAILABLE",
            "PLASMA__PLASMA_FUJIREBIO_NFL_GFAP_AVAILABLE",
            "PLASMA__AVAILABLE_PLASMA_BIOMARKER_COUNT",
        ],
    },

    "apoe": {
        "continuous_features": [],
        "categorical_features": [
            "APOE__APOE4_ALLELE_COUNT",
        ],
        "availability_columns": [
            "APOE__APOE_AVAILABLE",
        ],
    },

    "mri": {
        "continuous_features": [],
        "categorical_features": [],
        "path_columns": [
            "MRI__NORMALIZED_T1_NPY_PATH",
        ],
        "availability_columns": [
            "MRI_AVAILABLE",
        ],
    },
}


# ---------------------------------------------------------------------
# I validate that every column listed in the final schema exists in the
# participant-level multimodal master table.
# ---------------------------------------------------------------------
schema_validation_rows = []

for branch_name, branch_configuration in FINAL_BRANCH_SCHEMA.items():

    branch_column_groups = {
        "continuous_feature": branch_configuration.get(
            "continuous_features",
            [],
        ),
        "categorical_feature": branch_configuration.get(
            "categorical_features",
            [],
        ),
        "availability": branch_configuration.get(
            "availability_columns",
            [],
        ),
        "feature_level_availability": branch_configuration.get(
            "feature_level_availability_columns",
            [],
        ),
        "path": branch_configuration.get(
            "path_columns",
            [],
        ),
    }

    for column_role, columns in branch_column_groups.items():
        for column in columns:
            exists = column in multimodal_master_df.columns

            schema_validation_rows.append(
                {
                    "BRANCH": branch_name,
                    "COLUMN_ROLE": column_role,
                    "COLUMN": column,
                    "EXISTS": exists,
                    "DTYPE": (
                        str(multimodal_master_df[column].dtype)
                        if exists
                        else pd.NA
                    ),
                    "NON_MISSING": (
                        int(
                            multimodal_master_df[
                                column
                            ].notna().sum()
                        )
                        if exists
                        else pd.NA
                    ),
                    "MISSING_PERCENT": (
                        round(
                            100
                            * multimodal_master_df[
                                column
                            ].isna().mean(),
                            2,
                        )
                        if exists
                        else pd.NA
                    ),
                }
            )


final_branch_schema_validation_df = pd.DataFrame(
    schema_validation_rows
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    260,
):
    display(final_branch_schema_validation_df)


missing_schema_columns = (
    final_branch_schema_validation_df.loc[
        ~final_branch_schema_validation_df["EXISTS"]
    ]
)

if not missing_schema_columns.empty:
    display(missing_schema_columns)

    raise KeyError(
        "At least one column in the final branch schema is absent from "
        "the multimodal master table."
    )


# ---------------------------------------------------------------------
# I create one overall cognitive-functional availability indicator.
#
# The branch is available when at least one of ADAS, MMSE, or FAQ is
# available.
#
# I preserve the three original assessment-specific indicators because
# the later model and masking logic may still need to distinguish which
# assessment was observed.
# ---------------------------------------------------------------------
cognitive_availability_columns = (
    FINAL_BRANCH_SCHEMA[
        "cognitive_functional"
    ]["availability_columns"]
)

multimodal_master_df[
    "COGNITIVE_FUNCTIONAL_AVAILABLE"
] = (
    multimodal_master_df[
        cognitive_availability_columns
    ]
    .astype(bool)
    .any(axis=1)
)

FINAL_BRANCH_SCHEMA[
    "cognitive_functional"
]["branch_availability_column"] = (
    "COGNITIVE_FUNCTIONAL_AVAILABLE"
)

FINAL_BRANCH_SCHEMA[
    "demographics"
]["branch_availability_column"] = (
    "DEMOGRAPHICS_AVAILABLE"
)

FINAL_BRANCH_SCHEMA[
    "csf"
]["branch_availability_column"] = (
    "CSF__CSF_AVAILABLE"
)

FINAL_BRANCH_SCHEMA[
    "plasma"
]["branch_availability_column"] = (
    "PLASMA__PLASMA_AVAILABLE"
)

FINAL_BRANCH_SCHEMA[
    "apoe"
]["branch_availability_column"] = (
    "APOE__APOE_AVAILABLE"
)

FINAL_BRANCH_SCHEMA[
    "mri"
]["branch_availability_column"] = (
    "MRI_AVAILABLE"
)


# ---------------------------------------------------------------------
# I summarise the frozen branch dimensions and participant coverage.
# ---------------------------------------------------------------------
branch_summary_rows = []

for branch_name, branch_configuration in FINAL_BRANCH_SCHEMA.items():

    continuous_features = branch_configuration.get(
        "continuous_features",
        [],
    )

    categorical_features = branch_configuration.get(
        "categorical_features",
        [],
    )

    path_columns = branch_configuration.get(
        "path_columns",
        [],
    )

    branch_availability_column = (
        branch_configuration[
            "branch_availability_column"
        ]
    )

    available_count = int(
        multimodal_master_df[
            branch_availability_column
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    branch_summary_rows.append(
        {
            "BRANCH": branch_name,
            "CONTINUOUS_FEATURE_COUNT": len(
                continuous_features
            ),
            "CATEGORICAL_FEATURE_COUNT": len(
                categorical_features
            ),
            "PATH_COLUMN_COUNT": len(path_columns),
            "BRANCH_AVAILABILITY_COLUMN": (
                branch_availability_column
            ),
            "AVAILABLE_PARTICIPANTS": available_count,
            "UNAVAILABLE_PARTICIPANTS": (
                len(multimodal_master_df)
                - available_count
            ),
            "COVERAGE_PERCENT": round(
                100
                * available_count
                / len(multimodal_master_df),
                2,
            ),
        }
    )


final_branch_summary_df = pd.DataFrame(
    branch_summary_rows
)

with pd.option_context(
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    240,
):
    display(final_branch_summary_df)


# ---------------------------------------------------------------------
# I save the branch schema as JSON.
#
# This file will later be loaded by the model-training notebook so that
# the architecture uses exactly the same feature definitions.
# ---------------------------------------------------------------------
FINAL_BRANCH_SCHEMA_PATH = (
    MANIFEST_OUTPUT_DIR
    / "final_multimodal_branch_schema.json"
)

with open(
    FINAL_BRANCH_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        FINAL_BRANCH_SCHEMA,
        file,
        indent=2,
    )

if not FINAL_BRANCH_SCHEMA_PATH.is_file():
    raise FileNotFoundError(
        "The final multimodal branch schema was not saved:\n"
        f"{FINAL_BRANCH_SCHEMA_PATH}"
    )

print(
    "\nFinal multimodal branch structure defined successfully."
)

print(
    f"Saved branch schema: {FINAL_BRANCH_SCHEMA_PATH}"
)

# 25. Create branch-level and feature-level observation masks

The final branch structure is now fixed.

The next step is to create the masks that will tell the model which inputs are genuinely observed.

create two kinds of masks.

### 25.1. Branch-level masks

A branch-level mask records whether an entire modality branch is available for a participant.

The six branch masks are:

- demographics;
- cognitive and functional assessment;
- CSF;
- plasma;
- APOE;
- MRI.

These masks will later be used by the interaction pathway missing-modality logic and by modality dropout.

### 25.2. Feature-level masks

A feature-level mask records whether each individual scalar feature is observed.

This is necessary because a participant may have a modality available while still missing one specific feature. For example:

- CSF may be available while Aβ40 and the Aβ42/Aβ40 ratio are missing;
- plasma may contain p-tau217 but not one of the NfL or GFAP assay measurements;
- demographics may be available while one field is missing;
- the cognitive-functional branch may contain FAQ and ADAS but not MMSE.

For each scalar feature, the mask will use:

- `1` when the original value is observed;
- `0` when the original value is missing.

not fill or scale the feature values in this cell. The masks will be created directly from the unmodified participant-level master table.

The masks will later allow us to use a neutral numerical placeholder for tensor construction without pretending that the missing measurement was actually observed.

In [ ]:
# ---------------------------------------------------------------------
# I create branch-level and feature-level observation masks directly
# from the frozen multimodal master table.
#
# A mask value of:
#   1 = the branch or feature is observed;
#   0 = the branch or feature is unavailable or missing.
#
# I do not alter, fill, or scale the original feature values here.
# ---------------------------------------------------------------------

mask_master_df = multimodal_master_df[
    [
        "RID",
        "PTID",
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ]
].copy()


# ---------------------------------------------------------------------
# Branch-level masks
#
# These masks describe whether each complete modelling branch is
# available for a participant.
# ---------------------------------------------------------------------
BRANCH_MASK_COLUMNS = {}

for branch_name, branch_configuration in (
    FINAL_BRANCH_SCHEMA.items()
):
    source_availability_column = (
        branch_configuration[
            "branch_availability_column"
        ]
    )

    mask_column = (
        f"BRANCH_MASK__{branch_name.upper()}"
    )

    if source_availability_column not in (
        multimodal_master_df.columns
    ):
        raise KeyError(
            f"The branch availability column "
            f"{source_availability_column!r} is missing for "
            f"branch {branch_name!r}."
        )

    mask_master_df[mask_column] = (
        multimodal_master_df[
            source_availability_column
        ]
        .fillna(False)
        .astype(bool)
        .astype("int8")
    )

    BRANCH_MASK_COLUMNS[
        branch_name
    ] = mask_column


# ---------------------------------------------------------------------
# Feature-level masks
#
# I create one mask for every continuous and categorical scalar feature.
#
# MRI is represented by a path rather than a scalar feature vector, so
# its availability is already captured by the MRI branch mask.
# ---------------------------------------------------------------------
FEATURE_MASK_COLUMNS = {}

for branch_name, branch_configuration in (
    FINAL_BRANCH_SCHEMA.items()
):
    scalar_features = (
        branch_configuration.get(
            "continuous_features",
            [],
        )
        + branch_configuration.get(
            "categorical_features",
            [],
        )
    )

    FEATURE_MASK_COLUMNS[
        branch_name
    ] = {}

    for feature_column in scalar_features:
        if feature_column not in (
            multimodal_master_df.columns
        ):
            raise KeyError(
                f"The feature {feature_column!r} is missing from the "
                f"multimodal master table."
            )

        safe_feature_name = (
            feature_column
            .replace("__", "_")
            .replace(" ", "_")
        )

        mask_column = (
            f"FEATURE_MASK__{safe_feature_name}"
        )

        mask_master_df[mask_column] = (
            multimodal_master_df[
                feature_column
            ]
            .notna()
            .astype("int8")
        )

        FEATURE_MASK_COLUMNS[
            branch_name
        ][feature_column] = mask_column


# ---------------------------------------------------------------------
# I create compact mask summaries.
#
# These counts describe the number of observed participants for each
# branch and each scalar feature.
# ---------------------------------------------------------------------
branch_mask_summary_rows = []

for branch_name, mask_column in (
    BRANCH_MASK_COLUMNS.items()
):
    observed_count = int(
        mask_master_df[mask_column].sum()
    )

    branch_mask_summary_rows.append(
        {
            "BRANCH": branch_name,
            "MASK_COLUMN": mask_column,
            "OBSERVED_PARTICIPANTS": observed_count,
            "MISSING_PARTICIPANTS": (
                len(mask_master_df)
                - observed_count
            ),
            "OBSERVED_PERCENT": round(
                100
                * observed_count
                / len(mask_master_df),
                2,
            ),
        }
    )

branch_mask_summary_df = pd.DataFrame(
    branch_mask_summary_rows
)


feature_mask_summary_rows = []

for branch_name, feature_mapping in (
    FEATURE_MASK_COLUMNS.items()
):
    for feature_column, mask_column in (
        feature_mapping.items()
    ):
        observed_count = int(
            mask_master_df[mask_column].sum()
        )

        feature_mask_summary_rows.append(
            {
                "BRANCH": branch_name,
                "FEATURE": feature_column,
                "MASK_COLUMN": mask_column,
                "OBSERVED_VALUES": observed_count,
                "MISSING_VALUES": (
                    len(mask_master_df)
                    - observed_count
                ),
                "OBSERVED_PERCENT": round(
                    100
                    * observed_count
                    / len(mask_master_df),
                    2,
                ),
            }
        )

feature_mask_summary_df = pd.DataFrame(
    feature_mask_summary_rows
)


# ---------------------------------------------------------------------
# I confirm that all masks contain only 0 and 1.
# ---------------------------------------------------------------------
all_mask_columns = (
    list(BRANCH_MASK_COLUMNS.values())
    + [
        mask_column
        for feature_mapping in (
            FEATURE_MASK_COLUMNS.values()
        )
        for mask_column in (
            feature_mapping.values()
        )
    ]
)

invalid_mask_columns = []

for mask_column in all_mask_columns:
    observed_values = set(
        mask_master_df[
            mask_column
        ].dropna().unique()
    )

    if not observed_values.issubset({0, 1}):
        invalid_mask_columns.append(
            {
                "MASK_COLUMN": mask_column,
                "OBSERVED_VALUES": sorted(
                    observed_values
                ),
            }
        )

if invalid_mask_columns:
    invalid_mask_columns_df = pd.DataFrame(
        invalid_mask_columns
    )

    display(invalid_mask_columns_df)

    raise ValueError(
        "At least one observation mask contains values other than "
        "0 and 1."
    )


# ---------------------------------------------------------------------
# I confirm that the mask table preserves the complete authoritative
# participant set.
# ---------------------------------------------------------------------
if len(mask_master_df) != 2199:
    raise ValueError(
        "The observation-mask table should contain 2,199 participants."
    )

if mask_master_df["RID"].nunique() != 2199:
    raise ValueError(
        "The observation-mask table should contain 2,199 unique RIDs."
    )

if mask_master_df["RID"].duplicated().any():
    raise ValueError(
        "The observation-mask table contains duplicated RIDs."
    )


# ---------------------------------------------------------------------
# I display the summaries without hiding columns.
# ---------------------------------------------------------------------
print("BRANCH-LEVEL MASK SUMMARY")

with pd.option_context(
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    220,
):
    display(branch_mask_summary_df)


print("\nFEATURE-LEVEL MASK SUMMARY")

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    260,
):
    display(feature_mask_summary_df)


# ---------------------------------------------------------------------
# I save the mask definitions and participant-level mask table.
#
# The JSON files describe which mask belongs to each branch and feature.
# The CSV file contains the actual 0/1 masks for all participants.
# ---------------------------------------------------------------------
MASK_TABLE_PATH = (
    MANIFEST_OUTPUT_DIR
    / "multimodal_observation_masks_2199.csv"
)

BRANCH_MASK_SCHEMA_PATH = (
    MANIFEST_OUTPUT_DIR
    / "branch_mask_columns.json"
)

FEATURE_MASK_SCHEMA_PATH = (
    MANIFEST_OUTPUT_DIR
    / "feature_mask_columns.json"
)

mask_master_df.to_csv(
    MASK_TABLE_PATH,
    index=False,
)

with open(
    BRANCH_MASK_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        BRANCH_MASK_COLUMNS,
        file,
        indent=2,
    )

with open(
    FEATURE_MASK_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        FEATURE_MASK_COLUMNS,
        file,
        indent=2,
    )


for saved_path in [
    MASK_TABLE_PATH,
    BRANCH_MASK_SCHEMA_PATH,
    FEATURE_MASK_SCHEMA_PATH,
]:
    if not saved_path.is_file():
        raise FileNotFoundError(
            f"The mask output was not saved successfully:\n"
            f"{saved_path}"
        )


print(
    "\nBranch-level and feature-level observation masks were created "
    "successfully."
)

print(f"Saved mask table: {MASK_TABLE_PATH}")
print(f"Saved branch-mask schema: {BRANCH_MASK_SCHEMA_PATH}")
print(f"Saved feature-mask schema: {FEATURE_MASK_SCHEMA_PATH}")

# 26. Create fixed stratified five-fold participant assignments

create the participant splits that will be reused throughout model development.

use **five-fold stratified cross-validation** because the main experiments need stable class representation while making efficient use of the available cohort.

create two separate fold assignments.

### 26.1. Four-group fold assignment

This assignment contains all 2,199 authoritative participants and is stratified by:

- CN;
- AD;
- sMCI;
- pMCI.

This split can support:

- four-group classification;
- AD versus CN experiments;
- augmented classification experiments derived from the four-group cohort.

### 26.2. MCI prognosis fold assignment

This assignment contains only the 544 participants in the primary prognosis cohort:

- 295 sMCI participants;
- 249 pMCI participants.

It is stratified by the binary prognosis target.

The two assignments are saved separately because the four-group task and the prognosis task have different participant sets and different class labels.

Each participant receives exactly one fold number from 0 to 4.

For a given model run:

- one fold will be held out for validation or testing;
- the remaining folds will form the training set;
- all preprocessing statistics will be fitted using only the training folds.

No scaling, categorical encoding, or numerical placeholder filling is performed in this cell.

In [ ]:
# ---------------------------------------------------------------------
# I create fixed five-fold stratified participant assignments for:
#
#   1. the complete four-group authoritative cohort;
#   2. the primary pMCI-versus-sMCI prognosis cohort.
#
# These assignments are participant-level and deterministic because I
# use the global random seed defined at the beginning of the notebook.
# ---------------------------------------------------------------------

from sklearn.model_selection import StratifiedKFold


NUMBER_OF_FOLDS = 5

cross_validation_splitter = StratifiedKFold(
    n_splits=NUMBER_OF_FOLDS,
    shuffle=True,
    random_state=SEED,
)


# =====================================================================
# FOUR-GROUP FOLD ASSIGNMENT
# =====================================================================

four_group_fold_df = (
    authoritative_model_base_df[
        [
            "RID",
            "PTID",
            "CLINICAL_GROUP",
            "MCI_TRAJECTORY_LABEL",
            "MCI_PROGNOSIS_TARGET",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

four_group_fold_df["FOUR_GROUP_FOLD"] = -1

# I use a dummy feature matrix because StratifiedKFold only needs the
# participant labels to create stratified assignments.
four_group_dummy_features = np.zeros(
    shape=(len(four_group_fold_df), 1),
    dtype=np.float32,
)

four_group_labels = (
    four_group_fold_df["CLINICAL_GROUP"]
    .astype(str)
    .to_numpy()
)

for fold_number, (_, test_indices) in enumerate(
    cross_validation_splitter.split(
        four_group_dummy_features,
        four_group_labels,
    )
):
    four_group_fold_df.loc[
        test_indices,
        "FOUR_GROUP_FOLD",
    ] = fold_number


# ---------------------------------------------------------------------
# I verify that every four-group participant received exactly one valid
# fold assignment.
# ---------------------------------------------------------------------
if four_group_fold_df[
    "FOUR_GROUP_FOLD"
].eq(-1).any():
    raise ValueError(
        "At least one four-group participant did not receive a fold."
    )

if not set(
    four_group_fold_df[
        "FOUR_GROUP_FOLD"
    ].unique()
) == set(range(NUMBER_OF_FOLDS)):
    raise ValueError(
        "The four-group fold assignment does not contain exactly "
        "folds 0 through 4."
    )

if four_group_fold_df["RID"].duplicated().any():
    raise ValueError(
        "The four-group fold assignment contains duplicated RIDs."
    )

if len(four_group_fold_df) != 2199:
    raise ValueError(
        "The four-group fold assignment should contain 2,199 "
        "participants."
    )


# =====================================================================
# MCI PROGNOSIS FOLD ASSIGNMENT
# =====================================================================

mci_prognosis_fold_df = (
    authoritative_model_base_df.loc[
        authoritative_model_base_df[
            "CLINICAL_GROUP"
        ].isin(["sMCI", "pMCI"]),
        [
            "RID",
            "PTID",
            "CLINICAL_GROUP",
            "MCI_TRAJECTORY_LABEL",
            "MCI_PROGNOSIS_TARGET",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------
# I verify that the binary prognosis target is complete for the 544
# prognosis participants before constructing folds.
# ---------------------------------------------------------------------
if mci_prognosis_fold_df[
    "MCI_PROGNOSIS_TARGET"
].isna().any():
    missing_target_rows = (
        mci_prognosis_fold_df.loc[
            mci_prognosis_fold_df[
                "MCI_PROGNOSIS_TARGET"
            ].isna()
        ]
    )

    display(missing_target_rows)

    raise ValueError(
        "At least one pMCI or sMCI participant is missing the binary "
        "prognosis target."
    )

observed_prognosis_targets = set(
    mci_prognosis_fold_df[
        "MCI_PROGNOSIS_TARGET"
    ]
    .dropna()
    .astype(int)
    .unique()
)

if observed_prognosis_targets != {0, 1}:
    raise ValueError(
        "The MCI prognosis target must contain exactly the values "
        "0 and 1."
    )

mci_prognosis_fold_df["PROGNOSIS_FOLD"] = -1

mci_dummy_features = np.zeros(
    shape=(len(mci_prognosis_fold_df), 1),
    dtype=np.float32,
)

mci_labels = (
    mci_prognosis_fold_df[
        "MCI_PROGNOSIS_TARGET"
    ]
    .astype(int)
    .to_numpy()
)

for fold_number, (_, test_indices) in enumerate(
    cross_validation_splitter.split(
        mci_dummy_features,
        mci_labels,
    )
):
    mci_prognosis_fold_df.loc[
        test_indices,
        "PROGNOSIS_FOLD",
    ] = fold_number


# ---------------------------------------------------------------------
# I verify the prognosis fold assignment.
# ---------------------------------------------------------------------
if mci_prognosis_fold_df[
    "PROGNOSIS_FOLD"
].eq(-1).any():
    raise ValueError(
        "At least one prognosis participant did not receive a fold."
    )

if not set(
    mci_prognosis_fold_df[
        "PROGNOSIS_FOLD"
    ].unique()
) == set(range(NUMBER_OF_FOLDS)):
    raise ValueError(
        "The prognosis fold assignment does not contain exactly "
        "folds 0 through 4."
    )

if mci_prognosis_fold_df["RID"].duplicated().any():
    raise ValueError(
        "The prognosis fold assignment contains duplicated RIDs."
    )

if len(mci_prognosis_fold_df) != 544:
    raise ValueError(
        "The prognosis fold assignment should contain 544 participants."
    )


# =====================================================================
# FOLD-BALANCE SUMMARIES
# =====================================================================

four_group_fold_balance_df = (
    four_group_fold_df
    .groupby(
        [
            "FOUR_GROUP_FOLD",
            "CLINICAL_GROUP",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "CN",
            "AD",
            "sMCI",
            "pMCI",
        ],
        fill_value=0,
    )
    .reset_index()
)

four_group_fold_balance_df[
    "TOTAL"
] = four_group_fold_balance_df[
    [
        "CN",
        "AD",
        "sMCI",
        "pMCI",
    ]
].sum(axis=1)


mci_fold_balance_df = (
    mci_prognosis_fold_df
    .groupby(
        [
            "PROGNOSIS_FOLD",
            "CLINICAL_GROUP",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "sMCI",
            "pMCI",
        ],
        fill_value=0,
    )
    .reset_index()
)

mci_fold_balance_df[
    "TOTAL"
] = mci_fold_balance_df[
    [
        "sMCI",
        "pMCI",
    ]
].sum(axis=1)


print("FOUR-GROUP FOLD BALANCE")

display(
    four_group_fold_balance_df
)

print("\nMCI PROGNOSIS FOLD BALANCE")

display(
    mci_fold_balance_df
)


# =====================================================================
# SAVE FIXED FOLD ASSIGNMENTS
# =====================================================================

FOUR_GROUP_FOLD_PATH = (
    MANIFEST_OUTPUT_DIR
    / "four_group_stratified_5fold_assignments_2199.csv"
)

MCI_PROGNOSIS_FOLD_PATH = (
    MANIFEST_OUTPUT_DIR
    / "mci_prognosis_stratified_5fold_assignments_544.csv"
)

four_group_fold_df.to_csv(
    FOUR_GROUP_FOLD_PATH,
    index=False,
)

mci_prognosis_fold_df.to_csv(
    MCI_PROGNOSIS_FOLD_PATH,
    index=False,
)


for saved_path in [
    FOUR_GROUP_FOLD_PATH,
    MCI_PROGNOSIS_FOLD_PATH,
]:
    if not saved_path.is_file():
        raise FileNotFoundError(
            "A fold-assignment file was not saved successfully:\n"
            f"{saved_path}"
        )


print(
    "\nFixed stratified five-fold assignments were created "
    "successfully."
)

print(
    f"Saved four-group folds: {FOUR_GROUP_FOLD_PATH}"
)

print(
    f"Saved MCI prognosis folds: {MCI_PROGNOSIS_FOLD_PATH}"
)

### 26.3. Why the pMCI and sMCI folds are not forced to be exactly 50/50

The prognosis cohort is already relatively balanced:

- sMCI: 295 participants;
- pMCI: 249 participants.

This corresponds to approximately:

- 54.2% sMCI;
- 45.8% pMCI.

The purpose of stratification is to preserve this original class distribution in every fold, not to artificially force equal class counts.

As a result, each held-out fold contains approximately:

- 59 sMCI participants;
- 49-50 pMCI participants.

This closely matches the class proportions of the complete prognosis cohort.

Forcing every fold to contain exactly the same number of sMCI and pMCI participants would require one of the following:

- removing some sMCI participants;
- duplicating some pMCI participants;
- deliberately distorting the original cohort distribution.

These options would either discard valid data or introduce an artificial sampling distribution. Therefore, all 544 prognosis participants are retained and the natural class ratio is preserved.

The current imbalance is mild and does not justify changing the held-out folds. Any remaining class imbalance should be handled only within the training data, for example through:

- a class-weighted loss;
- weighted training sampling;
- validation-based decision-threshold selection.

The held-out folds should remain unchanged so that evaluation reflects the actual prognosis cohort.

Performance should also be reported using metrics that are informative under class imbalance, including:

- ROC-AUC;
- balanced accuracy;
- sensitivity;
- specificity;
- precision;
- recall;
- F1-score.

Therefore, the present fold distribution is intentional and methodologically appropriate rather than an error in the split.

# 27. Create training, validation, and test (held-out) roles for each outer fold

The five-fold assignments define which participants belong to each outer fold, but the model-training notebook still needs an explicit development structure for every cross-validation iteration.

For each outer fold:

- the selected outer fold will remain completely held out for final evaluation;
- the other four folds will form the development cohort;
- a stratified validation subset will be drawn from the development cohort;
- the remaining development participants will be used for training.

The validation subset is required for:

- early stopping;
- learning-rate scheduling;
- selecting the best checkpoint;
- choosing decision thresholds;
- comparing model variants without using the outer held-out fold.

The outer held-out fold must not influence any of these decisions.

create deterministic role assignments for both tasks:

1. the complete four-group cohort;
2. the primary sMCI-versus-pMCI prognosis cohort.

For every outer fold, each participant will receive exactly one role:

- `train`;
- `validation`;
- `test` - held out.

The validation subset will contain approximately 15% of the development cohort and will preserve the relevant class distribution. Since the development cohort represents approximately 80% of the complete data, the resulting overall proportions will be approximately:

- 68% training;
- 12% validation;
- 20% test.

These assignments will be fixed and saved before any scaling or categorical encoding is fitted.

### 27.1. Why the split is approximately 70\% training, 10\% validation, and 20\% test

In five-fold cross-validation, one complete outer fold is reserved as the test set in each iteration. Since one of five folds is held out, the test set contains approximately:

$$
\frac{1}{5}=0.20=20\%
$$

of the complete task-specific cohort.

The remaining four folds form the development cohort:

$$
1-0.20=0.80=80\%
$$

This development cohort must then be divided into training and validation subsets.

To obtain a validation set representing approximately 10\% of the complete cohort, the validation subset must contain 12.5\% of the 80\% development cohort:

$$
\frac{0.10}{0.80}=0.125
$$

Therefore, the validation fraction within the development cohort is set to:

```python
VALIDATION_FRACTION_WITHIN_DEVELOPMENT = 0.125

In [ ]:
# ---------------------------------------------------------------------
# I create explicit train, validation, and held-out participant roles
# for every outer cross-validation fold.
#
# The outer held-out fold is never used for:
#   - fitting numerical scalers;
#   - fitting categorical encodings;
#   - early stopping;
#   - checkpoint selection;
#   - threshold selection;
#   - model comparison.
#
# A stratified validation subset is drawn only from the remaining outer
# development participants.
# ---------------------------------------------------------------------

from sklearn.model_selection import train_test_split


VALIDATION_FRACTION_WITHIN_DEVELOPMENT = 0.15


def create_nested_role_assignments(
    fold_df: pd.DataFrame,
    outer_fold_column: str,
    stratification_column: str,
    task_name: str,
) -> pd.DataFrame:
    """
    I create one row per participant per outer fold.

    For each outer fold:
      - participants assigned to that fold become held-out;
      - all remaining participants form the development cohort;
      - a stratified validation subset is drawn from development;
      - all remaining development participants become training cases.

    The resulting table is deterministic because each outer fold uses
    the global seed plus its fold number.
    """

    required_columns = [
        "RID",
        "PTID",
        outer_fold_column,
        stratification_column,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in fold_df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"The {task_name} fold table is missing required columns: "
            + ", ".join(missing_columns)
        )

    assignment_tables = []

    for outer_fold in range(NUMBER_OF_FOLDS):

        test_mask = (
            fold_df[outer_fold_column]
            == outer_fold
        )

        test_df = (
            fold_df.loc[
                test_mask
            ]
            .copy()
        )

        development_df = (
            fold_df.loc[
                ~test_mask
            ]
            .copy()
        )

        # -------------------------------------------------------------
        # I split only the outer development cohort.
        #
        # Stratification preserves the relevant class distribution in
        # the training and validation subsets.
        # -------------------------------------------------------------
        train_indices, validation_indices = train_test_split(
            development_df.index.to_numpy(),
            test_size=VALIDATION_FRACTION_WITHIN_DEVELOPMENT,
            random_state=SEED + outer_fold,
            shuffle=True,
            stratify=development_df[
                stratification_column
            ],
        )

        role_df = fold_df.copy()

        role_df["OUTER_FOLD"] = outer_fold
        role_df["DATA_ROLE"] = pd.Series(
            pd.NA,
            index=role_df.index,
            dtype="string",
        )

        role_df.loc[
            train_indices,
            "DATA_ROLE",
        ] = "train"

        role_df.loc[
            validation_indices,
            "DATA_ROLE",
        ] = "validation"

        role_df.loc[
            test_df.index,
            "DATA_ROLE",
        ] = "test"

        # -------------------------------------------------------------
        # I verify that every participant receives exactly one role for
        # this outer fold.
        # -------------------------------------------------------------
        if role_df["DATA_ROLE"].isna().any():
            missing_role_rows = role_df.loc[
                role_df["DATA_ROLE"].isna(),
                [
                    "RID",
                    "PTID",
                    outer_fold_column,
                ],
            ]

            display(
                missing_role_rows.head(30)
            )

            raise ValueError(
                f"At least one {task_name} participant did not receive "
                f"a role for outer fold {outer_fold}."
            )

        observed_roles = set(
            role_df["DATA_ROLE"].unique()
        )

        if observed_roles != {
            "train",
            "validation",
            "test",
        }:
            raise ValueError(
                f"The {task_name} outer fold {outer_fold} does not "
                "contain all three expected roles."
            )

        role_df["TASK"] = task_name

        assignment_tables.append(
            role_df
        )

    complete_assignment_df = pd.concat(
        assignment_tables,
        ignore_index=True,
    )

    return complete_assignment_df


# =====================================================================
# FOUR-GROUP TRAIN / VALIDATION / HELD-OUT ROLES
# =====================================================================

four_group_role_assignments_df = (
    create_nested_role_assignments(
        fold_df=four_group_fold_df,
        outer_fold_column="FOUR_GROUP_FOLD",
        stratification_column="CLINICAL_GROUP",
        task_name="four_group",
    )
)


# =====================================================================
# MCI PROGNOSIS TRAIN / VALIDATION / HELD-OUT ROLES
# =====================================================================

mci_prognosis_role_assignments_df = (
    create_nested_role_assignments(
        fold_df=mci_prognosis_fold_df,
        outer_fold_column="PROGNOSIS_FOLD",
        stratification_column="MCI_PROGNOSIS_TARGET",
        task_name="mci_prognosis",
    )
)


# ---------------------------------------------------------------------
# I summarise participant counts by outer fold and role.
# ---------------------------------------------------------------------
four_group_role_count_df = (
    four_group_role_assignments_df
    .groupby(
        [
            "OUTER_FOLD",
            "DATA_ROLE",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "train",
            "validation",
            "test",
        ],
        fill_value=0,
    )
    .reset_index()
)

four_group_role_count_df[
    "TOTAL"
] = four_group_role_count_df[
    [
        "train",
        "validation",
        "test",
    ]
].sum(axis=1)


mci_role_count_df = (
    mci_prognosis_role_assignments_df
    .groupby(
        [
            "OUTER_FOLD",
            "DATA_ROLE",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "train",
            "validation",
            "test",
        ],
        fill_value=0,
    )
    .reset_index()
)

mci_role_count_df[
    "TOTAL"
] = mci_role_count_df[
    [
        "train",
        "validation",
        "test",
    ]
].sum(axis=1)


# ---------------------------------------------------------------------
# I also inspect class counts inside every role.
# ---------------------------------------------------------------------
four_group_role_balance_df = (
    four_group_role_assignments_df
    .groupby(
        [
            "OUTER_FOLD",
            "DATA_ROLE",
            "CLINICAL_GROUP",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "CN",
            "AD",
            "sMCI",
            "pMCI",
        ],
        fill_value=0,
    )
    .reset_index()
)

four_group_role_balance_df[
    "TOTAL"
] = four_group_role_balance_df[
    [
        "CN",
        "AD",
        "sMCI",
        "pMCI",
    ]
].sum(axis=1)


mci_role_balance_df = (
    mci_prognosis_role_assignments_df
    .groupby(
        [
            "OUTER_FOLD",
            "DATA_ROLE",
            "CLINICAL_GROUP",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "sMCI",
            "pMCI",
        ],
        fill_value=0,
    )
    .reset_index()
)

mci_role_balance_df[
    "TOTAL"
] = mci_role_balance_df[
    [
        "sMCI",
        "pMCI",
    ]
].sum(axis=1)


# ---------------------------------------------------------------------
# I validate that each participant appears exactly once per outer fold.
# ---------------------------------------------------------------------
expected_four_group_rows = (
    len(four_group_fold_df)
    * NUMBER_OF_FOLDS
)

expected_mci_rows = (
    len(mci_prognosis_fold_df)
    * NUMBER_OF_FOLDS
)

if len(
    four_group_role_assignments_df
) != expected_four_group_rows:
    raise ValueError(
        "The four-group role table has an unexpected number of rows."
    )

if len(
    mci_prognosis_role_assignments_df
) != expected_mci_rows:
    raise ValueError(
        "The MCI prognosis role table has an unexpected number of rows."
    )


four_group_duplicate_check = (
    four_group_role_assignments_df
    .duplicated(
        subset=[
            "OUTER_FOLD",
            "RID",
        ]
    )
)

if four_group_duplicate_check.any():
    raise ValueError(
        "A four-group participant appears more than once within an "
        "outer fold."
    )


mci_duplicate_check = (
    mci_prognosis_role_assignments_df
    .duplicated(
        subset=[
            "OUTER_FOLD",
            "RID",
        ]
    )
)

if mci_duplicate_check.any():
    raise ValueError(
        "An MCI prognosis participant appears more than once within an "
        "outer fold."
    )


# ---------------------------------------------------------------------
# I verify that the held-out participants exactly match the original
# outer-fold assignments.
# ---------------------------------------------------------------------
for outer_fold in range(NUMBER_OF_FOLDS):

    expected_four_group_test_rids = set(
        four_group_fold_df.loc[
            four_group_fold_df[
                "FOUR_GROUP_FOLD"
            ] == outer_fold,
            "RID",
        ]
    )

    observed_four_group_test_rids = set(
        four_group_role_assignments_df.loc[
            (
                four_group_role_assignments_df[
                    "OUTER_FOLD"
                ] == outer_fold
            )
            & (
                four_group_role_assignments_df[
                    "DATA_ROLE"
                ] == "test"
            ),
            "RID",
        ]
    )

    if (
        expected_four_group_test_rids
        != observed_four_group_test_rids
    ):
        raise ValueError(
            f"The four-group held-out RIDs do not match outer fold "
            f"{outer_fold}."
        )

    expected_mci_test_rids = set(
        mci_prognosis_fold_df.loc[
            mci_prognosis_fold_df[
                "PROGNOSIS_FOLD"
            ] == outer_fold,
            "RID",
        ]
    )

    observed_mci_test_rids = set(
        mci_prognosis_role_assignments_df.loc[
            (
                mci_prognosis_role_assignments_df[
                    "OUTER_FOLD"
                ] == outer_fold
            )
            & (
                mci_prognosis_role_assignments_df[
                    "DATA_ROLE"
                ] == "test"
            ),
            "RID",
        ]
    )

    if (
        expected_mci_test_rids
        != observed_mci_test_rids
    ):
        raise ValueError(
            f"The MCI prognosis held-out RIDs do not match outer fold "
            f"{outer_fold}."
        )


# ---------------------------------------------------------------------
# I display the role sizes and class distributions.
# ---------------------------------------------------------------------
print(
    "FOUR-GROUP ROLE COUNTS"
)

display(
    four_group_role_count_df
)

print(
    "\nFOUR-GROUP CLASS BALANCE BY ROLE"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.width",
    240,
):
    display(
        four_group_role_balance_df
    )


print(
    "\nMCI PROGNOSIS ROLE COUNTS"
)

display(
    mci_role_count_df
)

print(
    "\nMCI PROGNOSIS CLASS BALANCE BY ROLE"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.width",
    220,
):
    display(
        mci_role_balance_df
    )


# ---------------------------------------------------------------------
# I save the fixed role assignments.
# ---------------------------------------------------------------------
FOUR_GROUP_ROLE_ASSIGNMENT_PATH = (
    MANIFEST_OUTPUT_DIR
    / "four_group_5fold_train_validation_test_roles.csv"
)

MCI_PROGNOSIS_ROLE_ASSIGNMENT_PATH = (
    MANIFEST_OUTPUT_DIR
    / "mci_prognosis_5fold_train_validation_test_roles.csv"
)

four_group_role_assignments_df.to_csv(
    FOUR_GROUP_ROLE_ASSIGNMENT_PATH,
    index=False,
)

mci_prognosis_role_assignments_df.to_csv(
    MCI_PROGNOSIS_ROLE_ASSIGNMENT_PATH,
    index=False,
)


for saved_path in [
    FOUR_GROUP_ROLE_ASSIGNMENT_PATH,
    MCI_PROGNOSIS_ROLE_ASSIGNMENT_PATH,
]:
    if not saved_path.is_file():
        raise FileNotFoundError(
            "A train-validation-held-out role file was not saved:\n"
            f"{saved_path}"
        )


print(
    "\nFixed train, validation, and held-out assignments were created "
    "successfully for every outer fold."
)

print(
    f"Saved four-group role assignments: "
    f"{FOUR_GROUP_ROLE_ASSIGNMENT_PATH}"
)

print(
    f"Saved MCI prognosis role assignments: "
    f"{MCI_PROGNOSIS_ROLE_ASSIGNMENT_PATH}"
)

# 28. Fit fold-specific numerical scaling parameters using training participants only

The train, validation, and test roles are now fixed for every outer fold.

I can therefore calculate the numerical standardisation parameters without leaking information from validation or test participants.

For every outer fold and task, calculate the training-set mean and standard deviation for each continuous scalar feature:

$$
z_{ij}
=
\frac{x_{ij}-\mu_{j,\mathrm{train}}}
{\sigma_{j,\mathrm{train}}}
$$

where:

- $x_{ij}$ is participant $i$'s observed value for feature $j$;
- $\mu_{j,\mathrm{train}}$ is the mean calculated from observed training values only;
- $\sigma_{j,\mathrm{train}}$ is the corresponding training-set standard deviation.

Missing values will be excluded when calculating the mean and standard deviation. They will remain missing at this stage.

Separate scaling parameters are required for:

1. the four-group task;
2. the MCI prognosis task;
3. each of the five outer folds.

This is necessary because each outer fold has a different training subset. Reusing statistics calculated from the complete cohort would allow information from validation and test participants to influence preprocessing.

The following variables will not be standardised:

- sex;
- handedness;
- APOE ε4 allele count;
- branch and feature masks;
- labels and targets;
- MRI intensities, which have already been normalised separately.

If a feature has zero variance within a training subset, its effective standard deviation will be set to `1.0` for numerical safety and the feature will be explicitly flagged. This prevents division by zero without changing its constant standardised value.

In [ ]:
# ---------------------------------------------------------------------
# I collect the final continuous scalar features from the frozen branch
# schema.
#
# Only these columns will be standardised. Categorical features, masks,
# labels, targets, and MRI paths are excluded.
# ---------------------------------------------------------------------

CONTINUOUS_SCALAR_FEATURES = []

for branch_name, branch_configuration in (
    FINAL_BRANCH_SCHEMA.items()
):
    CONTINUOUS_SCALAR_FEATURES.extend(
        branch_configuration.get(
            "continuous_features",
            [],
        )
    )

# I preserve the configured order while removing accidental duplicates.
CONTINUOUS_SCALAR_FEATURES = list(
    dict.fromkeys(CONTINUOUS_SCALAR_FEATURES)
)

if not CONTINUOUS_SCALAR_FEATURES:
    raise ValueError(
        "No continuous scalar features were found in the final branch "
        "schema."
    )

missing_continuous_features = [
    feature
    for feature in CONTINUOUS_SCALAR_FEATURES
    if feature not in multimodal_master_df.columns
]

if missing_continuous_features:
    raise KeyError(
        "The multimodal master table is missing continuous features: "
        + ", ".join(missing_continuous_features)
    )


# ---------------------------------------------------------------------
# I create a feature-to-branch mapping for later reporting and loading.
# ---------------------------------------------------------------------
CONTINUOUS_FEATURE_TO_BRANCH = {}

for branch_name, branch_configuration in (
    FINAL_BRANCH_SCHEMA.items()
):
    for feature in branch_configuration.get(
        "continuous_features",
        [],
    ):
        CONTINUOUS_FEATURE_TO_BRANCH[
            feature
        ] = branch_name


# ---------------------------------------------------------------------
# I define a reusable function that fits one set of scaling parameters
# for every outer fold of one task.
#
# The function uses only rows whose DATA_ROLE is "train".
# ---------------------------------------------------------------------
def fit_fold_specific_scalers(
    role_assignments_df: pd.DataFrame,
    task_name: str,
) -> pd.DataFrame:
    """
    I calculate training-only mean and standard deviation values for all
    continuous scalar features in each outer fold.

    Missing feature values are excluded from the calculations.

    The returned table contains one row per:
      task × outer fold × continuous feature.
    """

    required_role_columns = [
        "RID",
        "OUTER_FOLD",
        "DATA_ROLE",
    ]

    missing_role_columns = [
        column
        for column in required_role_columns
        if column not in role_assignments_df.columns
    ]

    if missing_role_columns:
        raise KeyError(
            f"The {task_name} role table is missing required columns: "
            + ", ".join(missing_role_columns)
        )

    scaler_rows = []

    for outer_fold in range(NUMBER_OF_FOLDS):

        # -------------------------------------------------------------
        # I obtain only the training RIDs for the current outer fold.
        # Validation and test participants are excluded completely.
        # -------------------------------------------------------------
        training_rids = (
            role_assignments_df.loc[
                (
                    role_assignments_df[
                        "OUTER_FOLD"
                    ] == outer_fold
                )
                & (
                    role_assignments_df[
                        "DATA_ROLE"
                    ] == "train"
                ),
                "RID",
            ]
            .dropna()
            .astype(int)
            .unique()
        )

        if len(training_rids) == 0:
            raise ValueError(
                f"No training participants were found for {task_name}, "
                f"outer fold {outer_fold}."
            )

        training_feature_df = (
            multimodal_master_df.loc[
                multimodal_master_df[
                    "RID"
                ].isin(training_rids),
                [
                    "RID",
                    *CONTINUOUS_SCALAR_FEATURES,
                ],
            ]
            .copy()
        )

        if (
            training_feature_df["RID"].nunique()
            != len(training_rids)
        ):
            raise ValueError(
                f"The {task_name} training feature table for outer fold "
                f"{outer_fold} does not match the expected RID count."
            )

        # -------------------------------------------------------------
        # I fit each feature independently using only observed values.
        # -------------------------------------------------------------
        for feature in CONTINUOUS_SCALAR_FEATURES:

            observed_training_values = (
                pd.to_numeric(
                    training_feature_df[feature],
                    errors="coerce",
                )
                .dropna()
                .astype(float)
            )

            observed_count = len(
                observed_training_values
            )

            missing_count = (
                len(training_feature_df)
                - observed_count
            )

            if observed_count == 0:
                raise ValueError(
                    f"Feature {feature!r} has no observed training values "
                    f"for {task_name}, outer fold {outer_fold}."
                )

            training_mean = float(
                observed_training_values.mean()
            )

            # I use the sample standard deviation, matching the default
            # behaviour of pandas and sklearn StandardScaler's practical
            # role closely enough for our saved preprocessing schema.
            training_std = float(
                observed_training_values.std(
                    ddof=0
                )
            )

            zero_or_invalid_variance = (
                not np.isfinite(training_std)
                or training_std == 0.0
            )

            effective_std = (
                1.0
                if zero_or_invalid_variance
                else training_std
            )

            scaler_rows.append(
                {
                    "TASK": task_name,
                    "OUTER_FOLD": outer_fold,
                    "BRANCH": (
                        CONTINUOUS_FEATURE_TO_BRANCH[
                            feature
                        ]
                    ),
                    "FEATURE": feature,
                    "TRAIN_PARTICIPANT_COUNT": len(
                        training_feature_df
                    ),
                    "OBSERVED_TRAIN_VALUES": observed_count,
                    "MISSING_TRAIN_VALUES": missing_count,
                    "TRAIN_MEAN": training_mean,
                    "TRAIN_STD": training_std,
                    "EFFECTIVE_STD": effective_std,
                    "ZERO_OR_INVALID_VARIANCE": (
                        zero_or_invalid_variance
                    ),
                }
            )

    return pd.DataFrame(
        scaler_rows
    )


# =====================================================================
# FIT FOUR-GROUP SCALERS
# =====================================================================

four_group_scaler_parameters_df = (
    fit_fold_specific_scalers(
        role_assignments_df=(
            four_group_role_assignments_df
        ),
        task_name="four_group",
    )
)


# =====================================================================
# FIT MCI PROGNOSIS SCALERS
# =====================================================================

mci_prognosis_scaler_parameters_df = (
    fit_fold_specific_scalers(
        role_assignments_df=(
            mci_prognosis_role_assignments_df
        ),
        task_name="mci_prognosis",
    )
)


# ---------------------------------------------------------------------
# I combine both tasks into one long-format scaling table.
# ---------------------------------------------------------------------
all_scaler_parameters_df = pd.concat(
    [
        four_group_scaler_parameters_df,
        mci_prognosis_scaler_parameters_df,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------
# I validate the expected number of scaler rows.
#
# There should be:
#
#   2 tasks × 5 outer folds × number of continuous features.
# ---------------------------------------------------------------------
expected_scaler_rows = (
    2
    * NUMBER_OF_FOLDS
    * len(CONTINUOUS_SCALAR_FEATURES)
)

if len(all_scaler_parameters_df) != expected_scaler_rows:
    raise ValueError(
        "The scaling-parameter table has an unexpected number of rows. "
        f"Expected {expected_scaler_rows:,}, observed "
        f"{len(all_scaler_parameters_df):,}."
    )


# ---------------------------------------------------------------------
# I confirm that every fold contains one parameter row per feature.
# ---------------------------------------------------------------------
scaler_row_count_check_df = (
    all_scaler_parameters_df
    .groupby(
        [
            "TASK",
            "OUTER_FOLD",
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="FEATURE_PARAMETER_COUNT"
    )
)

scaler_row_count_check_df[
    "EXPECTED_FEATURE_COUNT"
] = len(
    CONTINUOUS_SCALAR_FEATURES
)

scaler_row_count_check_df[
    "MATCHES_EXPECTATION"
] = (
    scaler_row_count_check_df[
        "FEATURE_PARAMETER_COUNT"
    ]
    == scaler_row_count_check_df[
        "EXPECTED_FEATURE_COUNT"
    ]
)

display(
    scaler_row_count_check_df
)

if not scaler_row_count_check_df[
    "MATCHES_EXPECTATION"
].all():
    raise ValueError(
        "At least one task and outer-fold combination is missing scaling "
        "parameters for one or more continuous features."
    )


# ---------------------------------------------------------------------
# I inspect any zero-variance or invalid-standard-deviation features.
#
# These are not silently ignored. They are explicitly reported and use
# an effective standard deviation of 1.0.
# ---------------------------------------------------------------------
zero_variance_scaler_rows_df = (
    all_scaler_parameters_df.loc[
        all_scaler_parameters_df[
            "ZERO_OR_INVALID_VARIANCE"
        ]
    ]
    .copy()
)

if zero_variance_scaler_rows_df.empty:
    print(
        "\nNo continuous feature had zero or invalid variance in any "
        "training fold."
    )
else:
    print(
        "\nContinuous features with zero or invalid training variance:"
    )

    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.width",
        260,
    ):
        display(
            zero_variance_scaler_rows_df
        )


# ---------------------------------------------------------------------
# I display a compact example of the fitted parameters.
# ---------------------------------------------------------------------
print(
    "\nEXAMPLE TRAINING-ONLY SCALING PARAMETERS"
)

with pd.option_context(
    "display.max_rows",
    50,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    280,
):
    display(
        all_scaler_parameters_df.loc[
            (
                all_scaler_parameters_df[
                    "TASK"
                ] == "mci_prognosis"
            )
            & (
                all_scaler_parameters_df[
                    "OUTER_FOLD"
                ] == 0
            )
        ]
    )


# ---------------------------------------------------------------------
# I create a nested JSON structure for straightforward loading in the
# model-training notebook.
# ---------------------------------------------------------------------
scaler_parameters_json = {}

for task_name in (
    all_scaler_parameters_df["TASK"]
    .drop_duplicates()
):
    scaler_parameters_json[
        task_name
    ] = {}

    task_df = all_scaler_parameters_df.loc[
        all_scaler_parameters_df[
            "TASK"
        ] == task_name
    ]

    for outer_fold in range(NUMBER_OF_FOLDS):

        fold_df = task_df.loc[
            task_df[
                "OUTER_FOLD"
            ] == outer_fold
        ]

        scaler_parameters_json[
            task_name
        ][str(outer_fold)] = {
            row["FEATURE"]: {
                "branch": row["BRANCH"],
                "mean": float(
                    row["TRAIN_MEAN"]
                ),
                "std": float(
                    row["TRAIN_STD"]
                ),
                "effective_std": float(
                    row["EFFECTIVE_STD"]
                ),
                "observed_train_values": int(
                    row[
                        "OBSERVED_TRAIN_VALUES"
                    ]
                ),
                "missing_train_values": int(
                    row[
                        "MISSING_TRAIN_VALUES"
                    ]
                ),
                "zero_or_invalid_variance": bool(
                    row[
                        "ZERO_OR_INVALID_VARIANCE"
                    ]
                ),
            }
            for _, row in fold_df.iterrows()
        }


# ---------------------------------------------------------------------
# I save the fold-specific scaling parameters.
# ---------------------------------------------------------------------
SCALER_PARAMETER_CSV_PATH = (
    MANIFEST_OUTPUT_DIR
    / "fold_specific_training_only_scaler_parameters.csv"
)

SCALER_PARAMETER_JSON_PATH = (
    MANIFEST_OUTPUT_DIR
    / "fold_specific_training_only_scaler_parameters.json"
)

CONTINUOUS_FEATURE_SCHEMA_PATH = (
    MANIFEST_OUTPUT_DIR
    / "continuous_scalar_feature_columns.json"
)


all_scaler_parameters_df.to_csv(
    SCALER_PARAMETER_CSV_PATH,
    index=False,
)

with open(
    SCALER_PARAMETER_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        scaler_parameters_json,
        file,
        indent=2,
    )

with open(
    CONTINUOUS_FEATURE_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "continuous_scalar_features": (
                CONTINUOUS_SCALAR_FEATURES
            ),
            "feature_to_branch": (
                CONTINUOUS_FEATURE_TO_BRANCH
            ),
        },
        file,
        indent=2,
    )


for saved_path in [
    SCALER_PARAMETER_CSV_PATH,
    SCALER_PARAMETER_JSON_PATH,
    CONTINUOUS_FEATURE_SCHEMA_PATH,
]:
    if not saved_path.is_file():
        raise FileNotFoundError(
            "A scaling output was not saved successfully:\n"
            f"{saved_path}"
        )


print(
    "\nFold-specific training-only scaling parameters were fitted "
    "successfully."
)

print(
    f"Saved scaling table: "
    f"{SCALER_PARAMETER_CSV_PATH}"
)

print(
    f"Saved scaling JSON: "
    f"{SCALER_PARAMETER_JSON_PATH}"
)

print(
    f"Saved continuous-feature schema: "
    f"{CONTINUOUS_FEATURE_SCHEMA_PATH}"
)

# 29. Apply fold-specific scaling and create model-ready scalar input tables

The fold-specific scaling parameters have now been calculated using training participants only.

apply those parameters to the training, validation, and test participants within each outer fold.

For every continuous feature, calculate:

$$
z_{ij}
=
\frac{x_{ij}-\mu_{j,\mathrm{train}}}
{\sigma_{j,\mathrm{train}}}
$$

The same training-derived mean and standard deviation will be applied to all three roles:

- training;
- validation;
- test.

The validation and test values will not influence the transformation.

After standardisation, originally missing continuous values will be replaced with `0.0` only in the model-ready copy. This is a computational placeholder because zero represents the training mean in standardised space.

The corresponding feature mask remains separate:

- mask `1`: the standardised value was genuinely observed;
- mask `0`: the zero is a missing-value placeholder.

Therefore, this procedure does not claim that a missing clinical or biomarker measurement was observed or clinically imputed.

Categorical values will remain unchanged in this cell. Their embedding-compatible encoding will be defined separately.

A separate transformed table will be created for each:

- task;
- outer fold.

This is necessary because every outer fold has its own training-derived scaling parameters.

In [ ]:
# ---------------------------------------------------------------------
# I apply the fold-specific training-only scaling parameters to every
# participant role within each task and outer fold.
#
# The resulting tables contain:
#   - identifiers and targets;
#   - train / validation / test role;
#   - scaled continuous features;
#   - original categorical features;
#   - MRI path;
#   - branch-level masks;
#   - feature-level masks.
#
# Missing standardised values are replaced with 0.0 only after their
# observation masks have already been created and saved.
# ---------------------------------------------------------------------

FOLD_READY_OUTPUT_DIR = (
    MODEL_ROOT
    / "fold_ready_inputs"
)

FOLD_READY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# I collect categorical scalar features from the frozen branch schema.
# These remain in their cleaned original coding for now.
# ---------------------------------------------------------------------
CATEGORICAL_SCALAR_FEATURES = []

for branch_name, branch_configuration in (
    FINAL_BRANCH_SCHEMA.items()
):
    CATEGORICAL_SCALAR_FEATURES.extend(
        branch_configuration.get(
            "categorical_features",
            [],
        )
    )

CATEGORICAL_SCALAR_FEATURES = list(
    dict.fromkeys(
        CATEGORICAL_SCALAR_FEATURES
    )
)


# ---------------------------------------------------------------------
# I collect the MRI path column and all saved mask columns.
# ---------------------------------------------------------------------
MRI_PATH_COLUMNS = (
    FINAL_BRANCH_SCHEMA[
        "mri"
    ].get(
        "path_columns",
        [],
    )
)

ALL_BRANCH_MASK_COLUMNS = list(
    BRANCH_MASK_COLUMNS.values()
)

ALL_FEATURE_MASK_COLUMNS = [
    mask_column
    for branch_mapping in (
        FEATURE_MASK_COLUMNS.values()
    )
    for mask_column in (
        branch_mapping.values()
    )
]


# ---------------------------------------------------------------------
# I combine the original feature table with the previously created mask
# table by RID.
# ---------------------------------------------------------------------
mask_columns_for_merge = [
    "RID",
    *ALL_BRANCH_MASK_COLUMNS,
    *ALL_FEATURE_MASK_COLUMNS,
]

model_input_source_df = (
    multimodal_master_df.merge(
        mask_master_df[
            mask_columns_for_merge
        ],
        on="RID",
        how="left",
        validate="one_to_one",
    )
)

if len(model_input_source_df) != 2199:
    raise ValueError(
        "The model-input source table should contain 2,199 rows."
    )


# ---------------------------------------------------------------------
# I define a reusable function for one task.
# ---------------------------------------------------------------------
def create_fold_ready_task_tables(
    role_assignments_df: pd.DataFrame,
    task_name: str,
    participant_count: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    I create and save one model-ready scalar table per outer fold.

    Scaling uses the parameters fitted from that fold's training role.
    Missing continuous values become 0.0 only after scaling.
    """

    task_output_dir = (
        FOLD_READY_OUTPUT_DIR
        / task_name
    )

    task_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    saved_file_rows = []
    combined_fold_tables = []

    for outer_fold in range(
        NUMBER_OF_FOLDS
    ):

        # -------------------------------------------------------------
        # I select the participant roles for this outer fold.
        # -------------------------------------------------------------
        fold_roles_df = (
            role_assignments_df.loc[
                role_assignments_df[
                    "OUTER_FOLD"
                ] == outer_fold,
                [
                    "RID",
                    "DATA_ROLE",
                    "OUTER_FOLD",
                ],
            ]
            .copy()
        )

        if len(fold_roles_df) != participant_count:
            raise ValueError(
                f"{task_name}, fold {outer_fold}: expected "
                f"{participant_count:,} role rows but found "
                f"{len(fold_roles_df):,}."
            )

        # -------------------------------------------------------------
        # I obtain the complete participant inputs for this task.
        # An inner merge intentionally restricts the table to the task's
        # participant cohort.
        # -------------------------------------------------------------
        fold_ready_df = (
            fold_roles_df.merge(
                model_input_source_df,
                on="RID",
                how="inner",
                validate="one_to_one",
            )
        )

        if len(fold_ready_df) != participant_count:
            raise ValueError(
                f"{task_name}, fold {outer_fold}: the feature merge did "
                "not preserve the expected task cohort."
            )

        # -------------------------------------------------------------
        # I obtain only this task and outer fold's scaler parameters.
        # -------------------------------------------------------------
        fold_scaler_df = (
            all_scaler_parameters_df.loc[
                (
                    all_scaler_parameters_df[
                        "TASK"
                    ] == task_name
                )
                & (
                    all_scaler_parameters_df[
                        "OUTER_FOLD"
                    ] == outer_fold
                )
            ]
            .copy()
        )

        if (
            len(fold_scaler_df)
            != len(
                CONTINUOUS_SCALAR_FEATURES
            )
        ):
            raise ValueError(
                f"{task_name}, fold {outer_fold}: incomplete scaling "
                "parameter set."
            )

        # -------------------------------------------------------------
        # I apply each training-derived standardisation.
        #
        # The new model input column receives the suffix __Z.
        # I preserve the original unscaled columns in the source table,
        # but they are not copied into the final fold-ready table.
        # -------------------------------------------------------------
        scaled_feature_columns = []

        for _, parameter_row in (
            fold_scaler_df.iterrows()
        ):
            feature = parameter_row[
                "FEATURE"
            ]

            training_mean = float(
                parameter_row[
                    "TRAIN_MEAN"
                ]
            )

            effective_std = float(
                parameter_row[
                    "EFFECTIVE_STD"
                ]
            )

            scaled_column = (
                f"{feature}__Z"
            )

            numeric_values = pd.to_numeric(
                fold_ready_df[feature],
                errors="coerce",
            )

            fold_ready_df[
                scaled_column
            ] = (
                (
                    numeric_values
                    - training_mean
                )
                / effective_std
            )

            # ---------------------------------------------------------
            # Missing values become the neutral placeholder zero only
            # in this transformed model-ready column.
            # ---------------------------------------------------------
            fold_ready_df[
                scaled_column
            ] = (
                fold_ready_df[
                    scaled_column
                ]
                .fillna(0.0)
                .astype("float32")
            )

            scaled_feature_columns.append(
                scaled_column
            )

        # -------------------------------------------------------------
        # I construct the final ordered fold-ready table.
        # -------------------------------------------------------------
        identifier_and_target_columns = [
            "RID",
            "PTID",
            "CLINICAL_GROUP",
            "MCI_TRAJECTORY_LABEL",
            "MCI_PROGNOSIS_TARGET",
            "OUTER_FOLD",
            "DATA_ROLE",
        ]

        final_fold_columns = list(
            dict.fromkeys(
                identifier_and_target_columns
                + scaled_feature_columns
                + CATEGORICAL_SCALAR_FEATURES
                + MRI_PATH_COLUMNS
                + ALL_BRANCH_MASK_COLUMNS
                + ALL_FEATURE_MASK_COLUMNS
            )
        )

        missing_final_columns = [
            column
            for column in final_fold_columns
            if column not in fold_ready_df.columns
        ]

        if missing_final_columns:
            raise KeyError(
                f"{task_name}, fold {outer_fold}: final fold-ready "
                "columns are missing: "
                + ", ".join(
                    missing_final_columns
                )
            )

        final_fold_ready_df = (
            fold_ready_df[
                final_fold_columns
            ]
            .copy()
            .sort_values(
                [
                    "DATA_ROLE",
                    "RID",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        # -------------------------------------------------------------
        # I validate that transformed continuous values contain no NaN
        # or infinite values.
        # -------------------------------------------------------------
        scaled_values_array = (
            final_fold_ready_df[
                scaled_feature_columns
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        if np.isnan(
            scaled_values_array
        ).any():
            raise ValueError(
                f"{task_name}, fold {outer_fold}: NaN remained in the "
                "scaled model-ready features."
            )

        if not np.isfinite(
            scaled_values_array
        ).all():
            raise ValueError(
                f"{task_name}, fold {outer_fold}: non-finite scaled "
                "feature values were produced."
            )

        # -------------------------------------------------------------
        # I verify the role counts against the saved assignment table.
        # -------------------------------------------------------------
        expected_role_counts = (
            fold_roles_df[
                "DATA_ROLE"
            ]
            .value_counts()
            .sort_index()
        )

        observed_role_counts = (
            final_fold_ready_df[
                "DATA_ROLE"
            ]
            .value_counts()
            .sort_index()
        )

        if not expected_role_counts.equals(
            observed_role_counts
        ):
            raise ValueError(
                f"{task_name}, fold {outer_fold}: role counts changed "
                "during transformation."
            )

        # -------------------------------------------------------------
        # I save the fold-ready table.
        # -------------------------------------------------------------
        fold_output_path = (
            task_output_dir
            / (
                f"{task_name}_outer_fold_"
                f"{outer_fold}_model_ready.csv"
            )
        )

        final_fold_ready_df.to_csv(
            fold_output_path,
            index=False,
        )

        if not fold_output_path.is_file():
            raise FileNotFoundError(
                f"The fold-ready table was not saved:\n"
                f"{fold_output_path}"
            )

        saved_file_rows.append(
            {
                "TASK": task_name,
                "OUTER_FOLD": outer_fold,
                "OUTPUT_PATH": str(
                    fold_output_path
                ),
                "PARTICIPANT_ROWS": len(
                    final_fold_ready_df
                ),
                "TRAIN_ROWS": int(
                    (
                        final_fold_ready_df[
                            "DATA_ROLE"
                        ] == "train"
                    ).sum()
                ),
                "VALIDATION_ROWS": int(
                    (
                        final_fold_ready_df[
                            "DATA_ROLE"
                        ] == "validation"
                    ).sum()
                ),
                "TEST_ROWS": int(
                    (
                        final_fold_ready_df[
                            "DATA_ROLE"
                        ] == "test"
                    ).sum()
                ),
                "SCALED_FEATURE_COUNT": len(
                    scaled_feature_columns
                ),
                "CATEGORICAL_FEATURE_COUNT": len(
                    CATEGORICAL_SCALAR_FEATURES
                ),
                "BRANCH_MASK_COUNT": len(
                    ALL_BRANCH_MASK_COLUMNS
                ),
                "FEATURE_MASK_COUNT": len(
                    ALL_FEATURE_MASK_COLUMNS
                ),
            }
        )

        combined_fold_tables.append(
            final_fold_ready_df.assign(
                TASK=task_name
            )
        )

    return (
        pd.DataFrame(
            saved_file_rows
        ),
        pd.concat(
            combined_fold_tables,
            ignore_index=True,
        ),
    )


# =====================================================================
# CREATE FOUR-GROUP FOLD-READY TABLES
# =====================================================================

(
    four_group_fold_ready_inventory_df,
    four_group_all_fold_ready_df,
) = create_fold_ready_task_tables(
    role_assignments_df=(
        four_group_role_assignments_df
    ),
    task_name="four_group",
    participant_count=2199,
)


# =====================================================================
# CREATE MCI PROGNOSIS FOLD-READY TABLES
# =====================================================================

(
    mci_fold_ready_inventory_df,
    mci_all_fold_ready_df,
) = create_fold_ready_task_tables(
    role_assignments_df=(
        mci_prognosis_role_assignments_df
    ),
    task_name="mci_prognosis",
    participant_count=544,
)


# ---------------------------------------------------------------------
# I combine and display the saved-file inventory.
# ---------------------------------------------------------------------
fold_ready_inventory_df = pd.concat(
    [
        four_group_fold_ready_inventory_df,
        mci_fold_ready_inventory_df,
    ],
    ignore_index=True,
)

with pd.option_context(
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    280,
):
    display(
        fold_ready_inventory_df
    )


# ---------------------------------------------------------------------
# I save a combined inventory and transformed-feature schema.
# ---------------------------------------------------------------------
FOLD_READY_INVENTORY_PATH = (
    MANIFEST_OUTPUT_DIR
    / "fold_ready_model_input_inventory.csv"
)

TRANSFORMED_FEATURE_SCHEMA_PATH = (
    MANIFEST_OUTPUT_DIR
    / "transformed_scalar_feature_schema.json"
)

fold_ready_inventory_df.to_csv(
    FOLD_READY_INVENTORY_PATH,
    index=False,
)

with open(
    TRANSFORMED_FEATURE_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "scaled_continuous_features": {
                feature: f"{feature}__Z"
                for feature in (
                    CONTINUOUS_SCALAR_FEATURES
                )
            },
            "categorical_features_unencoded": (
                CATEGORICAL_SCALAR_FEATURES
            ),
            "mri_path_columns": (
                MRI_PATH_COLUMNS
            ),
            "branch_mask_columns": (
                BRANCH_MASK_COLUMNS
            ),
            "feature_mask_columns": (
                FEATURE_MASK_COLUMNS
            ),
            "missing_continuous_placeholder": 0.0,
            "missing_placeholder_interpretation": (
                "Training-mean placeholder in z-scored space; "
                "must be interpreted together with the feature mask."
            ),
        },
        file,
        indent=2,
    )


for saved_path in [
    FOLD_READY_INVENTORY_PATH,
    TRANSFORMED_FEATURE_SCHEMA_PATH,
]:
    if not saved_path.is_file():
        raise FileNotFoundError(
            "A fold-ready metadata file was not saved:\n"
            f"{saved_path}"
        )


print(
    "\nFold-specific model-ready scalar input tables were created "
    "successfully."
)

print(
    f"Saved fold-ready inventory: "
    f"{FOLD_READY_INVENTORY_PATH}"
)

print(
    f"Saved transformed feature schema: "
    f"{TRANSFORMED_FEATURE_SCHEMA_PATH}"
)

# 30. Encode categorical scalar features for model input

The continuous scalar features have now been standardised separately for every task and outer fold.

Three scalar features remain categorical:

- sex;
- handedness;
- APOE $\varepsilon4$ allele count.

These variables should not be z-score standardised because the numerical distance between their category codes does not represent a continuous clinical quantity.

For example, APOE $\varepsilon4$ allele counts of `0`, `1`, and `2` represent distinct genotype-risk categories. They should therefore be passed to the model as categorical indices and handled using embedding layers or another categorical encoder.

For every task and outer fold, I will:

1. construct the category mapping using the training subset only;
2. reserve index `0` for missing or previously unseen categories;
3. assign observed training categories indices beginning from `1`;
4. apply the same training-derived mapping to the validation and test subsets;
5. retain the original categorical columns for auditing;
6. add new model-input columns ending in `__IDX`.

The feature masks created previously remain authoritative:

- mask `1` means that the original categorical value was observed;
- mask `0` means that the categorical value was missing.

An encoded value of `0` must therefore be interpreted together with its corresponding feature mask.

This procedure prevents validation or test participants from influencing the preprocessing definitions used during model development.

In [ ]:
# ---------------------------------------------------------------------
# I create fold-specific categorical mappings using training rows only.
#
# Index 0 is reserved for:
#   - missing values;
#   - categories not observed in the corresponding training subset.
#
# Each observed training category receives an integer index beginning
# from 1.
# ---------------------------------------------------------------------

CATEGORICAL_ENCODING_OUTPUT_DIR = (
    MODEL_ROOT
    / "fold_ready_inputs_categorical_encoded"
)

CATEGORICAL_ENCODING_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# I define a helper that converts values into stable category keys.
#
# This avoids treating values such as 1 and 1.0 as different categories
# after CSV loading.
# ---------------------------------------------------------------------
def normalise_category_key(value):
    """
    I convert one categorical value into a stable string key.

    Missing values return None.
    Integer-like numeric values are stored without a decimal suffix.
    """

    if pd.isna(value):
        return None

    if isinstance(
        value,
        (
            int,
            np.integer,
        ),
    ):
        return str(
            int(value)
        )

    if isinstance(
        value,
        (
            float,
            np.floating,
        ),
    ):
        if float(value).is_integer():
            return str(
                int(value)
            )

        return str(
            float(value)
        )

    cleaned_value = str(
        value
    ).strip()

    if cleaned_value == "":
        return None

    return cleaned_value


# ---------------------------------------------------------------------
# I define a stable sorting rule for category keys.
#
# Numeric categories are sorted numerically. Other categories are sorted
# alphabetically.
# ---------------------------------------------------------------------
def category_sort_key(category_key):
    try:
        return (
            0,
            float(category_key),
        )
    except (
        TypeError,
        ValueError,
    ):
        return (
            1,
            str(category_key),
        )


# ---------------------------------------------------------------------
# I create one encoded table per task and outer fold.
# ---------------------------------------------------------------------
categorical_mapping_rows = []
categorical_encoded_inventory_rows = []

for _, inventory_row in (
    fold_ready_inventory_df.iterrows()
):

    task_name = inventory_row[
        "TASK"
    ]

    outer_fold = int(
        inventory_row[
            "OUTER_FOLD"
        ]
    )

    source_path = Path(
        inventory_row[
            "OUTPUT_PATH"
        ]
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "The fold-ready source table could not be found:\n"
            f"{source_path}"
        )

    fold_df = pd.read_csv(
        source_path
    )

    if len(fold_df) != int(
        inventory_row[
            "PARTICIPANT_ROWS"
        ]
    ):
        raise ValueError(
            f"{task_name}, fold {outer_fold}: the loaded participant "
            "count does not match the saved inventory."
        )

    training_df = fold_df.loc[
        fold_df[
            "DATA_ROLE"
        ] == "train"
    ].copy()

    if training_df.empty:
        raise ValueError(
            f"{task_name}, fold {outer_fold}: no training rows were "
            "found."
        )

    encoded_column_names = []

    for feature in (
        CATEGORICAL_SCALAR_FEATURES
    ):

        if feature not in fold_df.columns:
            raise KeyError(
                f"{task_name}, fold {outer_fold}: categorical feature "
                f"{feature!r} is missing."
            )

        # -------------------------------------------------------------
        # I identify the observed categories from training rows only.
        # -------------------------------------------------------------
        training_category_keys = (
            training_df[
                feature
            ]
            .map(
                normalise_category_key
            )
            .dropna()
            .unique()
            .tolist()
        )

        training_category_keys = sorted(
            training_category_keys,
            key=category_sort_key,
        )

        if len(
            training_category_keys
        ) == 0:
            raise ValueError(
                f"{task_name}, fold {outer_fold}: categorical feature "
                f"{feature!r} has no observed training categories."
            )

        # -------------------------------------------------------------
        # Index 0 is reserved for missing or unseen values.
        # -------------------------------------------------------------
        category_to_index = {
            category_key: category_index
            for category_index, category_key in enumerate(
                training_category_keys,
                start=1,
            )
        }

        encoded_column = (
            f"{feature}__IDX"
        )

        normalised_full_keys = (
            fold_df[
                feature
            ].map(
                normalise_category_key
            )
        )

        fold_df[
            encoded_column
        ] = (
            normalised_full_keys
            .map(
                category_to_index
            )
            .fillna(0)
            .astype(
                "int64"
            )
        )

        encoded_column_names.append(
            encoded_column
        )

        # -------------------------------------------------------------
        # I count missing and unseen values separately for reporting.
        # -------------------------------------------------------------
        missing_value_mask = (
            normalised_full_keys.isna()
        )

        unseen_value_mask = (
            normalised_full_keys.notna()
            & ~normalised_full_keys.isin(
                category_to_index.keys()
            )
        )

        for (
            category_key,
            category_index,
        ) in category_to_index.items():

            categorical_mapping_rows.append(
                {
                    "TASK": task_name,
                    "OUTER_FOLD": outer_fold,
                    "FEATURE": feature,
                    "CATEGORY_KEY": category_key,
                    "CATEGORY_INDEX": int(
                        category_index
                    ),
                    "TRAINING_DERIVED": True,
                }
            )

        categorical_mapping_rows.append(
            {
                "TASK": task_name,
                "OUTER_FOLD": outer_fold,
                "FEATURE": feature,
                "CATEGORY_KEY": (
                    "__MISSING_OR_UNSEEN__"
                ),
                "CATEGORY_INDEX": 0,
                "TRAINING_DERIVED": False,
            }
        )

        print(
            f"{task_name}, fold {outer_fold}, {feature}: "
            f"{len(category_to_index)} observed training categories, "
            f"{int(missing_value_mask.sum())} missing values, "
            f"{int(unseen_value_mask.sum())} unseen non-missing values."
        )

    # -----------------------------------------------------------------
    # I validate the encoded categorical columns.
    # -----------------------------------------------------------------
    for encoded_column in (
        encoded_column_names
    ):

        if fold_df[
            encoded_column
        ].isna().any():
            raise ValueError(
                f"{task_name}, fold {outer_fold}: NaN remained in "
                f"{encoded_column!r}."
            )

        if (
            fold_df[
                encoded_column
            ] < 0
        ).any():
            raise ValueError(
                f"{task_name}, fold {outer_fold}: negative categorical "
                f"indices were produced in {encoded_column!r}."
            )

    # -----------------------------------------------------------------
    # I save a new table rather than overwriting the previous fold-ready
    # table. This preserves the earlier stage for auditing.
    # -----------------------------------------------------------------
    task_output_dir = (
        CATEGORICAL_ENCODING_OUTPUT_DIR
        / task_name
    )

    task_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    encoded_output_path = (
        task_output_dir
        / (
            f"{task_name}_outer_fold_"
            f"{outer_fold}_categorical_encoded.csv"
        )
    )

    fold_df.to_csv(
        encoded_output_path,
        index=False,
    )

    if not encoded_output_path.is_file():
        raise FileNotFoundError(
            "The categorical-encoded table was not saved:\n"
            f"{encoded_output_path}"
        )

    categorical_encoded_inventory_rows.append(
        {
            "TASK": task_name,
            "OUTER_FOLD": outer_fold,
            "SOURCE_PATH": str(
                source_path
            ),
            "ENCODED_OUTPUT_PATH": str(
                encoded_output_path
            ),
            "PARTICIPANT_ROWS": len(
                fold_df
            ),
            "TRAIN_ROWS": int(
                (
                    fold_df[
                        "DATA_ROLE"
                    ] == "train"
                ).sum()
            ),
            "VALIDATION_ROWS": int(
                (
                    fold_df[
                        "DATA_ROLE"
                    ] == "validation"
                ).sum()
            ),
            "TEST_ROWS": int(
                (
                    fold_df[
                        "DATA_ROLE"
                    ] == "test"
                ).sum()
            ),
            "ENCODED_CATEGORICAL_FEATURE_COUNT": len(
                encoded_column_names
            ),
        }
    )


# ---------------------------------------------------------------------
# I combine the mapping and inventory records.
# ---------------------------------------------------------------------
categorical_mapping_df = pd.DataFrame(
    categorical_mapping_rows
)

categorical_encoded_inventory_df = pd.DataFrame(
    categorical_encoded_inventory_rows
)


# ---------------------------------------------------------------------
# I confirm that all ten expected fold tables were produced.
#
# There should be:
#   2 tasks × 5 outer folds = 10 files.
# ---------------------------------------------------------------------
expected_encoded_table_count = (
    2
    * NUMBER_OF_FOLDS
)

if (
    len(
        categorical_encoded_inventory_df
    )
    != expected_encoded_table_count
):
    raise ValueError(
        "The number of categorical-encoded fold tables is incorrect. "
        f"Expected {expected_encoded_table_count}, observed "
        f"{len(categorical_encoded_inventory_df)}."
    )


# ---------------------------------------------------------------------
# I display the resulting inventory and mappings.
# ---------------------------------------------------------------------
print(
    "\nCATEGORICAL-ENCODED FOLD INVENTORY"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    280,
):
    display(
        categorical_encoded_inventory_df
    )


print(
    "\nTRAINING-DERIVED CATEGORICAL MAPPINGS"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    280,
):
    display(
        categorical_mapping_df
    )


# ---------------------------------------------------------------------
# I save the categorical mappings in CSV and JSON formats.
# ---------------------------------------------------------------------
CATEGORICAL_MAPPING_CSV_PATH = (
    MANIFEST_OUTPUT_DIR
    / "fold_specific_training_only_categorical_mappings.csv"
)

CATEGORICAL_MAPPING_JSON_PATH = (
    MANIFEST_OUTPUT_DIR
    / "fold_specific_training_only_categorical_mappings.json"
)

CATEGORICAL_ENCODED_INVENTORY_PATH = (
    MANIFEST_OUTPUT_DIR
    / "categorical_encoded_fold_input_inventory.csv"
)


categorical_mapping_df.to_csv(
    CATEGORICAL_MAPPING_CSV_PATH,
    index=False,
)

categorical_encoded_inventory_df.to_csv(
    CATEGORICAL_ENCODED_INVENTORY_PATH,
    index=False,
)


categorical_mapping_json = {}

for task_name in (
    categorical_mapping_df[
        "TASK"
    ].drop_duplicates()
):

    categorical_mapping_json[
        task_name
    ] = {}

    for outer_fold in range(
        NUMBER_OF_FOLDS
    ):

        fold_mapping_df = (
            categorical_mapping_df.loc[
                (
                    categorical_mapping_df[
                        "TASK"
                    ] == task_name
                )
                & (
                    categorical_mapping_df[
                        "OUTER_FOLD"
                    ] == outer_fold
                )
            ]
        )

        categorical_mapping_json[
            task_name
        ][str(outer_fold)] = {}

        for feature in (
            CATEGORICAL_SCALAR_FEATURES
        ):

            feature_mapping_df = (
                fold_mapping_df.loc[
                    fold_mapping_df[
                        "FEATURE"
                    ] == feature
                ]
            )

            categorical_mapping_json[
                task_name
            ][str(outer_fold)][
                feature
            ] = {
                str(
                    row[
                        "CATEGORY_KEY"
                    ]
                ): int(
                    row[
                        "CATEGORY_INDEX"
                    ]
                )
                for _, row in (
                    feature_mapping_df.iterrows()
                )
            }


with open(
    CATEGORICAL_MAPPING_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        categorical_mapping_json,
        file,
        indent=2,
    )


for saved_path in [
    CATEGORICAL_MAPPING_CSV_PATH,
    CATEGORICAL_MAPPING_JSON_PATH,
    CATEGORICAL_ENCODED_INVENTORY_PATH,
]:
    if not saved_path.is_file():
        raise FileNotFoundError(
            "A categorical encoding output was not saved:\n"
            f"{saved_path}"
        )


print(
    "\nCategorical scalar features were encoded successfully using "
    "training-derived fold-specific mappings."
)

print(
    f"Saved categorical mapping table: "
    f"{CATEGORICAL_MAPPING_CSV_PATH}"
)

print(
    f"Saved categorical mapping JSON: "
    f"{CATEGORICAL_MAPPING_JSON_PATH}"
)

print(
    f"Saved encoded-input inventory: "
    f"{CATEGORICAL_ENCODED_INVENTORY_PATH}"
)

### 30.1. Interpreting categorical encoding

The categorical variables were converted into integer indices so that they can later be passed to embedding layers or another categorical encoder.

The category mappings were created separately for every task and outer fold using the **training subset only**. Validation and test participants did not influence the mapping.

#### 30.1.1. Meaning of `missing`

A value is **missing** when no valid value is available for that participant.

Examples:

- sex is unavailable;
- handedness is unavailable;
- APOE genotype is unavailable, so the APOE $\varepsilon4$ allele count cannot be determined.

Missing categorical values are assigned:

```text
encoded index = 0

# 31. Encode the task targets and create final task-ready input tables

The scalar input features are now standardised and the categorical predictors are encoded.

The remaining task-specific labels must now be converted into explicit numerical targets.

Two prediction tasks are retained:

### 31.1. Four-group classification

The authoritative clinical groups will use the following fixed mapping:

| Clinical group | Encoded target |
|---|---:|
| CN | `0` |
| sMCI | `1` |
| pMCI | `2` |
| AD | `3` |

This mapping is fixed from the clinical definition of the task. It is therefore not estimated from an individual training fold.

The ordering does not imply that the model should treat the four classes as a continuous or ordinal variable. The encoded values are class indices for a multiclass classification loss.

### 31.2. MCI prognosis classification

The authoritative prognosis target is already binary:

| MCI outcome | Encoded target |
|---|---:|
| sMCI | `0` |
| pMCI | `1` |

The existing `MCI_PROGNOSIS_TARGET` column will be retained and copied into a clearly named model-target column.

For every task and outer fold, create one final task-ready table containing:

- participant identifiers;
- the numerical target for that task;
- train, validation, or test role;
- scaled continuous predictors;
- encoded categorical predictors;
- MRI path;
- branch-availability masks;
- feature-availability masks.

The original clinical labels will also be retained for auditing and interpretation.

No participant will be removed during this step.

In [ ]:
# ---------------------------------------------------------------------
# I define the fixed target mappings for the two prediction tasks.
#
# These mappings describe the task labels themselves. They are not
# learned preprocessing parameters and therefore remain identical across
# all outer folds.
# ---------------------------------------------------------------------

FOUR_GROUP_TARGET_MAPPING = {
    "CN": 0,
    "sMCI": 1,
    "pMCI": 2,
    "AD": 3,
}

MCI_PROGNOSIS_TARGET_MAPPING = {
    "sMCI": 0,
    "pMCI": 1,
}


# ---------------------------------------------------------------------
# I identify the final model-input feature columns.
# ---------------------------------------------------------------------

SCALED_CONTINUOUS_COLUMNS = [
    f"{feature}__Z"
    for feature in CONTINUOUS_SCALAR_FEATURES
]

ENCODED_CATEGORICAL_COLUMNS = [
    f"{feature}__IDX"
    for feature in CATEGORICAL_SCALAR_FEATURES
]

FINAL_MODEL_FEATURE_COLUMNS = list(
    dict.fromkeys(
        SCALED_CONTINUOUS_COLUMNS
        + ENCODED_CATEGORICAL_COLUMNS
        + MRI_PATH_COLUMNS
        + ALL_BRANCH_MASK_COLUMNS
        + ALL_FEATURE_MASK_COLUMNS
    )
)


# ---------------------------------------------------------------------
# I create the output folder for the final task-ready tables.
# ---------------------------------------------------------------------

FINAL_TASK_READY_OUTPUT_DIR = (
    MODEL_ROOT
    / "final_task_ready_inputs"
)

FINAL_TASK_READY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# I define a reusable function that adds and validates the appropriate
# model target for one task.
# ---------------------------------------------------------------------

def create_final_task_ready_tables(
    encoded_inventory_df: pd.DataFrame,
    task_name: str,
) -> pd.DataFrame:
    """
    I create one final task-ready table for every outer fold.

    The function adds the numerical target appropriate for the selected
    task and preserves the original clinical labels for auditing.
    """

    task_inventory_df = (
        encoded_inventory_df.loc[
            encoded_inventory_df["TASK"] == task_name
        ]
        .copy()
        .sort_values("OUTER_FOLD")
        .reset_index(drop=True)
    )

    if len(task_inventory_df) != NUMBER_OF_FOLDS:
        raise ValueError(
            f"{task_name}: expected {NUMBER_OF_FOLDS} encoded fold "
            f"tables, but found {len(task_inventory_df)}."
        )

    task_output_dir = (
        FINAL_TASK_READY_OUTPUT_DIR
        / task_name
    )

    task_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_inventory_rows = []

    for _, inventory_row in task_inventory_df.iterrows():

        outer_fold = int(
            inventory_row["OUTER_FOLD"]
        )

        encoded_source_path = Path(
            inventory_row["ENCODED_OUTPUT_PATH"]
        )

        if not encoded_source_path.is_file():
            raise FileNotFoundError(
                "The categorical-encoded source table was not found:\n"
                f"{encoded_source_path}"
            )

        fold_df = pd.read_csv(
            encoded_source_path
        )

        # -------------------------------------------------------------
        # I add the task-specific numerical target.
        # -------------------------------------------------------------
        if task_name == "four_group":

            fold_df["MODEL_TARGET"] = (
                fold_df["CLINICAL_GROUP"]
                .map(FOUR_GROUP_TARGET_MAPPING)
            )

            expected_target_values = {
                0,
                1,
                2,
                3,
            }

            target_mapping_used = (
                FOUR_GROUP_TARGET_MAPPING
            )

        elif task_name == "mci_prognosis":

            # ---------------------------------------------------------
            # I restrict the task to the authoritative MCI prognosis
            # cohort and copy its already validated binary target.
            # ---------------------------------------------------------
            invalid_mci_groups = sorted(
                set(
                    fold_df["CLINICAL_GROUP"]
                    .dropna()
                    .unique()
                )
                - {
                    "sMCI",
                    "pMCI",
                }
            )

            if invalid_mci_groups:
                raise ValueError(
                    "The MCI prognosis table contains non-MCI clinical "
                    "groups: "
                    + ", ".join(invalid_mci_groups)
                )

            fold_df["MODEL_TARGET"] = pd.to_numeric(
                fold_df["MCI_PROGNOSIS_TARGET"],
                errors="coerce",
            )

            expected_target_values = {
                0,
                1,
            }

            target_mapping_used = (
                MCI_PROGNOSIS_TARGET_MAPPING
            )

        else:
            raise ValueError(
                f"Unsupported task name: {task_name!r}"
            )

        # -------------------------------------------------------------
        # I confirm that every participant has a valid target.
        # -------------------------------------------------------------
        if fold_df["MODEL_TARGET"].isna().any():

            missing_target_rows = int(
                fold_df["MODEL_TARGET"]
                .isna()
                .sum()
            )

            raise ValueError(
                f"{task_name}, fold {outer_fold}: "
                f"{missing_target_rows} participants have no valid "
                "model target."
            )

        fold_df["MODEL_TARGET"] = (
            fold_df["MODEL_TARGET"]
            .astype("int64")
        )

        observed_target_values = set(
            fold_df["MODEL_TARGET"]
            .unique()
            .tolist()
        )

        if observed_target_values != expected_target_values:
            raise ValueError(
                f"{task_name}, fold {outer_fold}: unexpected target "
                f"values. Expected {sorted(expected_target_values)}, "
                f"observed {sorted(observed_target_values)}."
            )

        # -------------------------------------------------------------
        # I verify that the numerical targets agree with the original
        # clinical labels.
        # -------------------------------------------------------------
        expected_target_from_label = (
            fold_df["CLINICAL_GROUP"]
            .map(target_mapping_used)
        )

        target_mismatch_mask = (
            expected_target_from_label
            .astype("int64")
            != fold_df["MODEL_TARGET"]
        )

        if target_mismatch_mask.any():
            raise ValueError(
                f"{task_name}, fold {outer_fold}: numerical targets do "
                "not agree with the authoritative clinical labels."
            )

        # -------------------------------------------------------------
        # I ensure that all required final feature columns are present.
        # -------------------------------------------------------------
        missing_model_feature_columns = [
            column
            for column in FINAL_MODEL_FEATURE_COLUMNS
            if column not in fold_df.columns
        ]

        if missing_model_feature_columns:
            raise KeyError(
                f"{task_name}, fold {outer_fold}: final model feature "
                "columns are missing: "
                + ", ".join(missing_model_feature_columns)
            )

        # -------------------------------------------------------------
        # I define a clear and stable final column order.
        # -------------------------------------------------------------
        final_identifier_and_target_columns = [
            "RID",
            "PTID",
            "CLINICAL_GROUP",
            "MCI_TRAJECTORY_LABEL",
            "MCI_PROGNOSIS_TARGET",
            "MODEL_TARGET",
            "OUTER_FOLD",
            "DATA_ROLE",
        ]

        final_columns = list(
            dict.fromkeys(
                final_identifier_and_target_columns
                + FINAL_MODEL_FEATURE_COLUMNS
            )
        )

        final_task_ready_df = (
            fold_df[
                final_columns
            ]
            .copy()
            .sort_values(
                [
                    "DATA_ROLE",
                    "RID",
                ]
            )
            .reset_index(drop=True)
        )

        # -------------------------------------------------------------
        # I confirm that the role counts have not changed.
        # -------------------------------------------------------------
        observed_train_rows = int(
            (
                final_task_ready_df["DATA_ROLE"]
                == "train"
            ).sum()
        )

        observed_validation_rows = int(
            (
                final_task_ready_df["DATA_ROLE"]
                == "validation"
            ).sum()
        )

        observed_test_rows = int(
            (
                final_task_ready_df["DATA_ROLE"]
                == "test"
            ).sum()
        )

        expected_train_rows = int(
            inventory_row["TRAIN_ROWS"]
        )

        expected_validation_rows = int(
            inventory_row["VALIDATION_ROWS"]
        )

        expected_test_rows = int(
            inventory_row["TEST_ROWS"]
        )

        if (
            observed_train_rows != expected_train_rows
            or observed_validation_rows
            != expected_validation_rows
            or observed_test_rows != expected_test_rows
        ):
            raise ValueError(
                f"{task_name}, fold {outer_fold}: role counts changed "
                "while creating the final task-ready table."
            )

        # -------------------------------------------------------------
        # I save the final table.
        # -------------------------------------------------------------
        final_output_path = (
            task_output_dir
            / (
                f"{task_name}_outer_fold_"
                f"{outer_fold}_final_task_ready.csv"
            )
        )

        final_task_ready_df.to_csv(
            final_output_path,
            index=False,
        )

        if not final_output_path.is_file():
            raise FileNotFoundError(
                "The final task-ready table was not saved:\n"
                f"{final_output_path}"
            )

        # -------------------------------------------------------------
        # I record the class balance separately for each role.
        # -------------------------------------------------------------
        role_target_counts = (
            final_task_ready_df
            .groupby(
                [
                    "DATA_ROLE",
                    "MODEL_TARGET",
                ],
                observed=True,
            )
            .size()
            .unstack(
                fill_value=0
            )
        )

        final_inventory_rows.append(
            {
                "TASK": task_name,
                "OUTER_FOLD": outer_fold,
                "SOURCE_PATH": str(
                    encoded_source_path
                ),
                "FINAL_OUTPUT_PATH": str(
                    final_output_path
                ),
                "PARTICIPANT_ROWS": len(
                    final_task_ready_df
                ),
                "TRAIN_ROWS": observed_train_rows,
                "VALIDATION_ROWS": (
                    observed_validation_rows
                ),
                "TEST_ROWS": observed_test_rows,
                "TARGET_CLASS_COUNT": len(
                    observed_target_values
                ),
                "MODEL_FEATURE_COLUMN_COUNT": len(
                    FINAL_MODEL_FEATURE_COLUMNS
                ),
                "SCALED_CONTINUOUS_COUNT": len(
                    SCALED_CONTINUOUS_COLUMNS
                ),
                "ENCODED_CATEGORICAL_COUNT": len(
                    ENCODED_CATEGORICAL_COLUMNS
                ),
                "BRANCH_MASK_COUNT": len(
                    ALL_BRANCH_MASK_COLUMNS
                ),
                "FEATURE_MASK_COUNT": len(
                    ALL_FEATURE_MASK_COLUMNS
                ),
                "ROLE_TARGET_COUNTS": (
                    role_target_counts
                    .to_dict()
                ),
            }
        )

    return pd.DataFrame(
        final_inventory_rows
    )


# =====================================================================
# CREATE FINAL FOUR-GROUP TABLES
# =====================================================================

four_group_final_inventory_df = (
    create_final_task_ready_tables(
        encoded_inventory_df=(
            categorical_encoded_inventory_df
        ),
        task_name="four_group",
    )
)


# =====================================================================
# CREATE FINAL MCI PROGNOSIS TABLES
# =====================================================================

mci_prognosis_final_inventory_df = (
    create_final_task_ready_tables(
        encoded_inventory_df=(
            categorical_encoded_inventory_df
        ),
        task_name="mci_prognosis",
    )
)


# ---------------------------------------------------------------------
# I combine the final inventory records.
# ---------------------------------------------------------------------

final_task_ready_inventory_df = pd.concat(
    [
        four_group_final_inventory_df,
        mci_prognosis_final_inventory_df,
    ],
    ignore_index=True,
)


print(
    "FINAL TASK-READY INPUT INVENTORY"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
    "display.width",
    300,
):
    display(
        final_task_ready_inventory_df
    )


# ---------------------------------------------------------------------
# I save the final target mappings.
# ---------------------------------------------------------------------

TARGET_MAPPING_PATH = (
    MANIFEST_OUTPUT_DIR
    / "task_target_mappings.json"
)

with open(
    TARGET_MAPPING_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "four_group": {
                "target_column": "MODEL_TARGET",
                "mapping": (
                    FOUR_GROUP_TARGET_MAPPING
                ),
                "interpretation": (
                    "Nominal multiclass indices; the numerical order "
                    "does not define an ordinal regression target."
                ),
            },
            "mci_prognosis": {
                "target_column": "MODEL_TARGET",
                "mapping": (
                    MCI_PROGNOSIS_TARGET_MAPPING
                ),
                "positive_class": "pMCI",
            },
        },
        file,
        indent=2,
    )


# ---------------------------------------------------------------------
# I save the final table inventory.
# ---------------------------------------------------------------------

FINAL_TASK_READY_INVENTORY_PATH = (
    MANIFEST_OUTPUT_DIR
    / "final_task_ready_input_inventory.csv"
)

final_task_ready_inventory_df.to_csv(
    FINAL_TASK_READY_INVENTORY_PATH,
    index=False,
)


# ---------------------------------------------------------------------
# I save the definitive model-input column schema.
# ---------------------------------------------------------------------

FINAL_MODEL_INPUT_SCHEMA_PATH = (
    MANIFEST_OUTPUT_DIR
    / "final_model_input_schema.json"
)

with open(
    FINAL_MODEL_INPUT_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "identifier_columns": [
                "RID",
                "PTID",
            ],
            "audit_label_columns": [
                "CLINICAL_GROUP",
                "MCI_TRAJECTORY_LABEL",
                "MCI_PROGNOSIS_TARGET",
            ],
            "model_target_column": (
                "MODEL_TARGET"
            ),
            "split_columns": [
                "OUTER_FOLD",
                "DATA_ROLE",
            ],
            "scaled_continuous_columns": (
                SCALED_CONTINUOUS_COLUMNS
            ),
            "encoded_categorical_columns": (
                ENCODED_CATEGORICAL_COLUMNS
            ),
            "mri_path_columns": (
                MRI_PATH_COLUMNS
            ),
            "branch_mask_columns": (
                ALL_BRANCH_MASK_COLUMNS
            ),
            "feature_mask_columns": (
                ALL_FEATURE_MASK_COLUMNS
            ),
            "complete_model_feature_columns": (
                FINAL_MODEL_FEATURE_COLUMNS
            ),
        },
        file,
        indent=2,
    )


for saved_path in [
    TARGET_MAPPING_PATH,
    FINAL_TASK_READY_INVENTORY_PATH,
    FINAL_MODEL_INPUT_SCHEMA_PATH,
]:
    if not saved_path.is_file():
        raise FileNotFoundError(
            "A final task-preparation output was not saved:\n"
            f"{saved_path}"
        )


print(
    "\nTask targets were encoded and the final task-ready input tables "
    "were created successfully."
)

print(
    f"Saved target mappings: "
    f"{TARGET_MAPPING_PATH}"
)

print(
    f"Saved final input inventory: "
    f"{FINAL_TASK_READY_INVENTORY_PATH}"
)

print(
    f"Saved final model-input schema: "
    f"{FINAL_MODEL_INPUT_SCHEMA_PATH}"
)